In [ ]:
!pip install pandas numpy matplotlib pillow kagglehub -q

In [ ]:
!pip install ipympl -q

In [ ]:
%matplotlib widget

In [ ]:
%matplotlib notebook

In [ ]:
import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "divyanshusingh369/complete-pokemon-library-32k-images-and-csv"
)

dataset_path = Path(dataset_path)

print("Dataset descargado en:")
print(dataset_path)

In [ ]:
import pandas as pd
import numpy as np
import re

url_stats = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"
df = pd.read_csv(url_stats)

df.head()

In [ ]:
def limpiar_columna(col):
    col = str(col).strip().lower()
    col = col.replace(" ", "_")
    col = col.replace(".", "")
    col = col.replace("#", "number")
    return col

df.columns = [limpiar_columna(c) for c in df.columns]
print(df.columns.tolist())

In [ ]:
stat_cols = ["hp", "attack", "defense", "sp_atk", "sp_def", "speed"]

for c in stat_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["total_stats"] = df[stat_cols].sum(axis=1)

df[["number", "name", "type_1", "generation", "total_stats"]].head()

In [ ]:
from pathlib import Path

image_extensions = ["*.png", "*.jpg", "*.jpeg", "*.webp"]
image_files = []

for ext in image_extensions:
    image_files.extend(dataset_path.rglob(ext))

print("Total imágenes encontradas:", len(image_files))

In [ ]:
def normalizar_nombre(texto):
    texto = str(texto).lower().strip()
    texto = texto.replace("♀", "f")
    texto = texto.replace("♂", "m")
    texto = texto.replace(".", "")
    texto = texto.replace("'", "")
    texto = texto.replace("-", "")
    texto = texto.replace(" ", "")
    texto = re.sub(r"[^a-z0-9]", "", texto)
    return texto

def score_imagen(path):
    ruta = str(path).lower()
    score = 0
    
    if "sprite" in ruta:
        score += 10
    if "front" in ruta:
        score += 8
    if "default" in ruta:
        score += 5
    if "official" in ruta:
        score -= 3
    if path.suffix.lower() == ".png":
        score += 2
        
    return score

In [ ]:
image_map = {}

for img in image_files:
    posibles_keys = [
        normalizar_nombre(img.stem),
        normalizar_nombre(img.parent.name)
    ]
    
    score = score_imagen(img)
    
    for key in posibles_keys:
        if key == "":
            continue
        
        if key not in image_map:
            image_map[key] = (img, score)
        else:
            old_path, old_score = image_map[key]
            if score > old_score:
                image_map[key] = (img, score)

image_map = {k: v[0] for k, v in image_map.items()}

print("Claves de imagen indexadas:", len(image_map))

In [ ]:
def encontrar_imagen(nombre):
    key = normalizar_nombre(nombre)

    if key in image_map:
        return image_map[key]

    for k, path in image_map.items():
        if key == k:
            return path

    for k, path in image_map.items():
        if key in k or k in key:
            return path

    return None

df["image_path"] = df["name"].apply(encontrar_imagen)

print("Con imagen:", df["image_path"].notna().sum())
print("Sin imagen:", df["image_path"].isna().sum())

In [ ]:
plot_df = df.dropna(subset=["name", "type_1", "generation", "total_stats", "image_path"]).copy()

plot_df["generation"] = pd.to_numeric(plot_df["generation"], errors="coerce")
plot_df["total_stats"] = pd.to_numeric(plot_df["total_stats"], errors="coerce")

plot_df = plot_df.dropna(subset=["generation", "total_stats"]).copy()
plot_df["generation"] = plot_df["generation"].astype(int)

plot_df = plot_df[
    (plot_df["generation"] >= 1) &
    (plot_df["generation"] <= 9)
].copy()

plot_df[["name", "type_1", "generation", "total_stats"]].head()

In [ ]:
grupos = []

for (tipo, generacion), grupo in plot_df.groupby(["type_1", "generation"]):
    promedio = grupo["total_stats"].mean()
    
    grupo = grupo.copy()
    grupo["distancia_promedio"] = (grupo["total_stats"] - promedio).abs()
    
    representante = grupo.sort_values("distancia_promedio").iloc[0]
    
    grupos.append({
        "type_1": tipo,
        "generation": generacion,
        "avg_total_stats": promedio,
        "pokemon_name": representante["name"],
        "pokemon_total": representante["total_stats"],
        "image_path": representante["image_path"]
    })

summary_df = pd.DataFrame(grupos)

summary_df.head(20)

In [ ]:
type_order = (
    summary_df.groupby("type_1")["avg_total_stats"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

type_order

In [ ]:
type_order = [
    "Fire", "Water", "Grass", "Electric", "Psychic",
    "Dragon", "Normal", "Ghost", "Fighting", "Ice",
    "Rock", "Ground", "Dark", "Steel", "Poison",
    "Bug", "Fairy", "Flying"
]

type_order = [t for t in type_order if t in summary_df["type_1"].unique()]

In [ ]:
type_to_y = {tipo: i for i, tipo in enumerate(type_order)}
summary_df["type_y"] = summary_df["type_1"].map(type_to_y)

summary_df = summary_df.dropna(subset=["type_y"]).copy()
summary_df["type_y"] = summary_df["type_y"].astype(int)

summary_df.head()

In [ ]:
type_colors = {
    "Fire": "#e76f51",
    "Water": "#3a86ff",
    "Grass": "#43aa8b",
    "Electric": "#f9c74f",
    "Psychic": "#d16ba5",
    "Dragon": "#6a4c93",
    "Normal": "#8d99ae",
    "Ghost": "#5a189a",
    "Fighting": "#9d0208",
    "Ice": "#4cc9f0",
    "Rock": "#9c6644",
    "Ground": "#bc6c25",
    "Dark": "#495057",
    "Steel": "#6c757d",
    "Poison": "#9d4edd",
    "Bug": "#90be6d",
    "Fairy": "#ff99c8",
    "Flying": "#89c2d9"
}

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import Axes3D, proj3d
from PIL import Image
from matplotlib.lines import Line2D

In [ ]:
class AnnotationBbox3D(AnnotationBbox):
    def __init__(self, imagebox, xyz, *args, **kwargs):
        super().__init__(imagebox, (0, 0), *args, **kwargs)
        self.xyz = xyz

    def draw(self, renderer):
        x2, y2, _ = proj3d.proj_transform(
            self.xyz[0], self.xyz[1], self.xyz[2], self.axes.get_proj()
        )
        self.xy = (x2, y2)
        super().draw(renderer)

In [ ]:
def cargar_sprite(path, size=(48, 48)):
    img = Image.open(path).convert("RGBA")
    img.thumbnail(size)
    return np.array(img)

In [ ]:
fig = plt.figure(figsize=(16, 10))
ax = fig.add_subplot(111, projection="3d")

ax.set_title("Scatter 3D de Pokémon: Promedio de Estadísticas por Tipo y Generación", pad=20)

# Puntos guía invisibles o pequeños
for _, row in summary_df.iterrows():
    color = type_colors.get(row["type_1"], "gray")
    ax.scatter(
        row["generation"],
        row["type_y"],
        row["avg_total_stats"],
        color=color,
        s=10,
        alpha=0.25
    )

# Agregar sprite en cada coordenada
for _, row in summary_df.iterrows():
    try:
        sprite = cargar_sprite(row["image_path"], size=(42, 42))
        imagebox = OffsetImage(sprite, zoom=1)
        
        ab = AnnotationBbox3D(
            imagebox,
            (
                row["generation"],
                row["type_y"],
                row["avg_total_stats"]
            ),
            frameon=False,
            pad=0
        )
        ax.add_artist(ab)
    except:
        pass

# Ejes
ax.set_xlabel("Generación", labelpad=12)
ax.set_ylabel("Tipo", labelpad=18)
ax.set_zlabel("Promedio de estadísticas", labelpad=12)

ax.set_xticks(sorted(summary_df["generation"].unique()))
ax.set_yticks(range(len(type_order)))
ax.set_yticklabels(type_order)

# Vista parecida a tu imagen
ax.view_init(elev=16, azim=-68)

# Ajustar rangos
ax.set_xlim(summary_df["generation"].min() - 0.5, summary_df["generation"].max() + 0.5)
ax.set_ylim(-0.5, len(type_order) - 0.5)
ax.set_zlim(
    summary_df["avg_total_stats"].min() - 20,
    summary_df["avg_total_stats"].max() + 20
)

# Leyenda
legend_elements = []
for tipo in type_order:
    legend_elements.append(
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            label=tipo,
            markerfacecolor=type_colors.get(tipo, "gray"),
            markersize=8
        )
    )

ax.legend(
    handles=legend_elements,
    title="Tipo",
    bbox_to_anchor=(1.18, 0.95),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
!pip install pandas numpy plotly pillow kagglehub -q

In [ ]:
import kagglehub
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from PIL import Image
from io import BytesIO
import base64
import re

# =====================================================
# 1. Descargar dataset de imágenes de Kaggle
# =====================================================

dataset_path = kagglehub.dataset_download(
    "divyanshusingh369/complete-pokemon-library-32k-images-and-csv"
)

dataset_path = Path(dataset_path)

print("Dataset de imágenes:")
print(dataset_path)

# =====================================================
# 2. Cargar datos Pokémon con generación, tipo y stats
# =====================================================

url_stats = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"

df = pd.read_csv(url_stats)

def limpiar_columna(col):
    col = str(col).strip().lower()
    col = col.replace(" ", "_")
    col = col.replace(".", "")
    col = col.replace("#", "number")
    return col

df.columns = [limpiar_columna(c) for c in df.columns]

print("Columnas:")
print(df.columns.tolist())

# =====================================================
# 3. Calcular total/promedio de estadísticas
# =====================================================

stat_cols = [
    "hp",
    "attack",
    "defense",
    "sp_atk",
    "sp_def",
    "speed"
]

for col in stat_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["total_stats"] = df[stat_cols].sum(axis=1)

df = df.dropna(subset=["name", "type_1", "generation", "total_stats"]).copy()

df["generation"] = pd.to_numeric(df["generation"], errors="coerce")
df["generation"] = df["generation"].astype(int)

df.head()

In [ ]:
# =====================================================
# 4. Buscar imágenes en el dataset de Kaggle
# =====================================================

image_extensions = ["*.png", "*.jpg", "*.jpeg", "*.webp"]

image_files = []

for ext in image_extensions:
    image_files.extend(dataset_path.rglob(ext))

print("Imágenes encontradas:", len(image_files))


def normalizar_nombre(texto):
    texto = str(texto).lower().strip()
    texto = texto.replace("♀", "f")
    texto = texto.replace("♂", "m")
    texto = texto.replace(".", "")
    texto = texto.replace("'", "")
    texto = texto.replace("-", "")
    texto = texto.replace(" ", "")
    texto = re.sub(r"[^a-z0-9]", "", texto)
    return texto


def score_imagen(path):
    ruta = str(path).lower()
    score = 0

    if "sprite" in ruta:
        score += 10

    if "front" in ruta:
        score += 8

    if "default" in ruta:
        score += 6

    if "official" in ruta:
        score -= 5

    if path.suffix.lower() == ".png":
        score += 2

    return score


image_map = {}

for img in image_files:
    posibles_keys = [
        normalizar_nombre(img.stem),
        normalizar_nombre(img.parent.name)
    ]

    score = score_imagen(img)

    for key in posibles_keys:
        if key == "":
            continue

        if key not in image_map:
            image_map[key] = (img, score)
        else:
            old_path, old_score = image_map[key]

            if score > old_score:
                image_map[key] = (img, score)

image_map = {k: v[0] for k, v in image_map.items()}

print("Imágenes indexadas:", len(image_map))


def encontrar_imagen(nombre):
    key = normalizar_nombre(nombre)

    if key in image_map:
        return image_map[key]

    for k, path in image_map.items():
        if key in k or k in key:
            return path

    return None


df["image_path"] = df["name"].apply(encontrar_imagen)

print("Pokémon con imagen:", df["image_path"].notna().sum())
print("Pokémon sin imagen:", df["image_path"].isna().sum())

df[["name", "type_1", "generation", "total_stats", "image_path"]].head()

In [ ]:
# =====================================================
# 5. Preparar datos
# =====================================================

plot_df = df.dropna(
    subset=["name", "type_1", "generation", "total_stats", "image_path"]
).copy()

# Usamos solo generaciones 1 a 6 para que se parezca más a tu imagen
plot_df = plot_df[
    (plot_df["generation"] >= 1) &
    (plot_df["generation"] <= 6)
].copy()

# Tipos principales como en tu imagen
type_order = [
    "Fire",
    "Water",
    "Grass",
    "Electric",
    "Psychic",
    "Dragon",
    "Normal",
    "Ghost",
    "Fighting",
    "Ice"
]

plot_df = plot_df[plot_df["type_1"].isin(type_order)].copy()

# =====================================================
# 6. Agrupar por tipo y generación
# =====================================================

grupos = []

for (tipo, gen), grupo in plot_df.groupby(["type_1", "generation"]):
    promedio = grupo["total_stats"].mean()

    grupo = grupo.copy()
    grupo["distancia"] = (grupo["total_stats"] - promedio).abs()

    # Pokémon más cercano al promedio de ese grupo
    representante = grupo.sort_values("distancia").iloc[0]

    grupos.append({
        "type_1": tipo,
        "generation": gen,
        "avg_stats": promedio,
        "pokemon": representante["name"],
        "pokemon_stats": representante["total_stats"],
        "image_path": representante["image_path"]
    })

summary_df = pd.DataFrame(grupos)

type_to_num = {tipo: i for i, tipo in enumerate(type_order)}
summary_df["type_num"] = summary_df["type_1"].map(type_to_num)

summary_df = summary_df.dropna(subset=["type_num"]).copy()
summary_df["type_num"] = summary_df["type_num"].astype(int)

summary_df.head()

In [ ]:
# =====================================================
# 7. Convertir imágenes a base64
# =====================================================

def imagen_base64(path, size=(46, 46)):
    try:
        img = Image.open(path).convert("RGBA")
        img.thumbnail(size)

        buffer = BytesIO()
        img.save(buffer, format="PNG")

        encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")

        return "data:image/png;base64," + encoded

    except:
        return None


summary_df["sprite_base64"] = summary_df["image_path"].apply(imagen_base64)
summary_df = summary_df.dropna(subset=["sprite_base64"]).reset_index(drop=True)

print("Sprites que se mostrarán:", len(summary_df))

In [ ]:
# =====================================================
# 8. Proyección 3D simulada
# =====================================================

# Generación = eje horizontal
# Tipo = profundidad
# Promedio = altura

summary_df["x_base"] = summary_df["generation"]
summary_df["y_base"] = summary_df["avg_stats"]

# Estos valores controlan la sensación de profundidad
PROFUNDIDAD_X = -0.13
PROFUNDIDAD_Y = -23

summary_df["x_plot"] = summary_df["x_base"] + summary_df["type_num"] * PROFUNDIDAD_X
summary_df["y_plot"] = summary_df["y_base"] + summary_df["type_num"] * PROFUNDIDAD_Y

# Colores por tipo
type_colors = {
    "Fire": "#e76f51",
    "Water": "#3a86ff",
    "Grass": "#2a9d8f",
    "Electric": "#f4c430",
    "Psychic": "#d16ba5",
    "Dragon": "#6a4c93",
    "Normal": "#8d99ae",
    "Ghost": "#5a189a",
    "Fighting": "#9d0208",
    "Ice": "#4cc9f0"
}

# =====================================================
# 9. Crear gráfico
# =====================================================

fig = go.Figure()

# Líneas de fondo por tipo
for tipo in type_order:
    temp = summary_df[summary_df["type_1"] == tipo].copy()

    if len(temp) == 0:
        continue

    fig.add_trace(
        go.Scatter(
            x=temp["x_plot"],
            y=temp["y_plot"],
            mode="lines",
            line=dict(
                color="rgba(180,180,180,0.35)",
                width=1
            ),
            hoverinfo="skip",
            showlegend=False
        )
    )

# Puntos invisibles para hover
fig.add_trace(
    go.Scatter(
        x=summary_df["x_plot"],
        y=summary_df["y_plot"],
        mode="markers",
        marker=dict(
            size=36,
            color="rgba(0,0,0,0)"
        ),
        customdata=np.stack(
            [
                summary_df["pokemon"],
                summary_df["type_1"],
                summary_df["generation"],
                summary_df["avg_stats"].round(2)
            ],
            axis=-1
        ),
        hovertemplate=
            "<b>Pokémon:</b> %{customdata[0]}<br>" +
            "<b>Tipo:</b> %{customdata[1]}<br>" +
            "<b>Generación:</b> %{customdata[2]}<br>" +
            "<b>Promedio de estadísticas:</b> %{customdata[3]}<br>" +
            "<extra></extra>",
        showlegend=False
    )
)

# Agregar cada sprite
for _, row in summary_df.iterrows():
    fig.add_layout_image(
        dict(
            source=row["sprite_base64"],
            x=row["x_plot"],
            y=row["y_plot"],
            xref="x",
            yref="y",
            sizex=0.28,
            sizey=25,
            xanchor="center",
            yanchor="middle",
            layer="above"
        )
    )

# Leyenda falsa por tipo
for tipo in type_order:
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(
                size=10,
                color=type_colors.get(tipo, "gray")
            ),
            name=tipo
        )
    )

# =====================================================
# 10. Ejes y etiquetas
# =====================================================

# Etiquetas de tipo a la izquierda
annotations = []

for tipo in type_order:
    type_num = type_to_num[tipo]

    x_label = 0.55 + type_num * PROFUNDIDAD_X
    y_label = summary_df["avg_stats"].min() - 10 + type_num * PROFUNDIDAD_Y

    annotations.append(
        dict(
            x=x_label,
            y=y_label,
            text=tipo,
            showarrow=False,
            xanchor="right",
            font=dict(size=11, color="black")
        )
    )

annotations.append(
    dict(
        x=0.25,
        y=summary_df["avg_stats"].min() - 80,
        text="<b>Tipo</b>",
        showarrow=False,
        font=dict(size=14, color="black")
    )
)

fig.update_layout(
    title="<b>Scatter 3D de Pokémon: Promedio de Estadísticas por Tipo y Generación</b>",
    width=1050,
    height=720,
    plot_bgcolor="white",
    paper_bgcolor="white",
    annotations=annotations,
    legend=dict(
        title="Tipo inicial / símbolo",
        x=1.03,
        y=0.88,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    margin=dict(l=80, r=190, t=80, b=80),
    xaxis=dict(
        title="Generación",
        tickmode="array",
        tickvals=[1, 2, 3, 4, 5, 6],
        ticktext=["1", "2", "3", "4", "5", "6"],
        range=[0.2, 6.7],
        showgrid=True,
        gridcolor="rgba(200,200,200,0.5)",
        zeroline=False
    ),
    yaxis=dict(
        title="Promedio de estadísticas",
        range=[
            summary_df["y_plot"].min() - 45,
            summary_df["y_plot"].max() + 55
        ],
        showgrid=True,
        gridcolor="rgba(200,200,200,0.5)",
        zeroline=False
    )
)

import plotly.io as pio

pio.renderers.default = "browser"

fig.show()

In [ ]:
!pip install pandas numpy plotly pillow kagglehub requests -q

In [ ]:
import kagglehub
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from PIL import Image
from io import BytesIO
import base64
import requests
import re

# =========================================================
# 1. DESCARGAR DATASET DE IMÁGENES
# =========================================================

dataset_path = kagglehub.dataset_download(
    "divyanshusingh369/complete-pokemon-library-32k-images-and-csv"
)
dataset_path = Path(dataset_path)

print("Dataset de imágenes:")
print(dataset_path)

# =========================================================
# 2. CARGAR DATOS POKÉMON DESDE INTERNET
#    (incluye id, nombre, tipo y stats base)
# =========================================================

url = "https://raw.githubusercontent.com/fanzeyi/pokemon.json/master/pokedex.json"
data = requests.get(url).json()

rows = []

for p in data:
    pid = p.get("id")
    name = p.get("name", {}).get("english")
    types = p.get("type", [])
    base = p.get("base", {})

    if not pid or not name or not types or not base:
        continue

    hp = base.get("HP", np.nan)
    attack = base.get("Attack", np.nan)
    defense = base.get("Defense", np.nan)
    sp_atk = base.get("Sp. Attack", np.nan)
    sp_def = base.get("Sp. Defense", np.nan)
    speed = base.get("Speed", np.nan)

    total_stats = np.nansum([hp, attack, defense, sp_atk, sp_def, speed])

    rows.append({
        "number": pid,
        "name": name,
        "type_1": types[0],
        "hp": hp,
        "attack": attack,
        "defense": defense,
        "sp_atk": sp_atk,
        "sp_def": sp_def,
        "speed": speed,
        "total_stats": total_stats
    })

df = pd.DataFrame(rows)

# =========================================================
# 3. CALCULAR GENERACIÓN DESDE EL NÚMERO POKÉDEX
# =========================================================

def obtener_generacion(numero):
    if numero <= 151:
        return 1
    elif numero <= 251:
        return 2
    elif numero <= 386:
        return 3
    elif numero <= 493:
        return 4
    elif numero <= 649:
        return 5
    elif numero <= 721:
        return 6
    elif numero <= 809:
        return 7
    elif numero <= 905:
        return 8
    else:
        return 9

df["generation"] = df["number"].apply(obtener_generacion)

# =========================================================
# 4. BUSCAR IMÁGENES DEL DATASET DE KAGGLE
# =========================================================

image_extensions = ["*.png", "*.jpg", "*.jpeg", "*.webp"]
image_files = []

for ext in image_extensions:
    image_files.extend(dataset_path.rglob(ext))

print("Imágenes encontradas:", len(image_files))

def normalizar_nombre(texto):
    texto = str(texto).lower().strip()
    texto = texto.replace("♀", "f")
    texto = texto.replace("♂", "m")
    texto = texto.replace(".", "")
    texto = texto.replace("'", "")
    texto = texto.replace("-", "")
    texto = texto.replace(" ", "")
    texto = re.sub(r"[^a-z0-9]", "", texto)
    return texto

def score_imagen(path):
    ruta = str(path).lower()
    score = 0

    if "sprite" in ruta:
        score += 10
    if "front" in ruta:
        score += 8
    if "default" in ruta:
        score += 6
    if "official" in ruta:
        score -= 4
    if path.suffix.lower() == ".png":
        score += 2

    return score

image_map = {}

for img in image_files:
    keys = [
        normalizar_nombre(img.stem),
        normalizar_nombre(img.parent.name)
    ]

    score = score_imagen(img)

    for key in keys:
        if not key:
            continue

        if key not in image_map:
            image_map[key] = (img, score)
        else:
            old_img, old_score = image_map[key]
            if score > old_score:
                image_map[key] = (img, score)

image_map = {k: v[0] for k, v in image_map.items()}

def encontrar_imagen(nombre):
    key = normalizar_nombre(nombre)

    if key in image_map:
        return image_map[key]

    for k, path in image_map.items():
        if key == k:
            return path

    for k, path in image_map.items():
        if key in k or k in key:
            return path

    return None

df["image_path"] = df["name"].apply(encontrar_imagen)

print("Pokémon con imagen:", df["image_path"].notna().sum())
print("Pokémon sin imagen:", df["image_path"].isna().sum())

# =========================================================
# 5. FILTRAR SOLO LOS TIPOS Y GENERACIONES DE LA REFERENCIA
# =========================================================

type_order = [
    "Fire",
    "Water",
    "Grass",
    "Electric",
    "Psychic",
    "Dragon",
    "Normal",
    "Ghost",
    "Fighting",
    "Ice"
]

plot_df = df.copy()

plot_df = plot_df[
    (plot_df["generation"] >= 1) &
    (plot_df["generation"] <= 7) &
    (plot_df["type_1"].isin(type_order)) &
    (plot_df["image_path"].notna())
].copy()

# =========================================================
# 6. AGRUPAR POR TIPO Y GENERACIÓN
#    Y ELEGIR EL POKÉMON MÁS CERCANO AL PROMEDIO
# =========================================================

grupos = []

for tipo in type_order:
    for gen in range(1, 8):
        grupo = plot_df[
            (plot_df["type_1"] == tipo) &
            (plot_df["generation"] == gen)
        ].copy()

        if len(grupo) == 0:
            continue

        promedio = grupo["total_stats"].mean()
        grupo["distancia"] = (grupo["total_stats"] - promedio).abs()

        representante = grupo.sort_values("distancia").iloc[0]

        grupos.append({
            "type_1": tipo,
            "generation": gen,
            "avg_stats": promedio,
            "pokemon": representante["name"],
            "pokemon_stats": representante["total_stats"],
            "image_path": representante["image_path"]
        })

summary_df = pd.DataFrame(grupos)

# =========================================================
# 7. CONVERTIR IMAGEN A BASE64
# =========================================================

def imagen_base64(path, size=(48, 48)):
    try:
        img = Image.open(path).convert("RGBA")
        img.thumbnail(size)

        buffer = BytesIO()
        img.save(buffer, format="PNG")

        encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
        return "data:image/png;base64," + encoded

    except:
        return None

summary_df["sprite_base64"] = summary_df["image_path"].apply(imagen_base64)
summary_df = summary_df.dropna(subset=["sprite_base64"]).reset_index(drop=True)

# =========================================================
# 8. CREAR PROYECCIÓN 3D FIJA
#    ESTO HACE QUE SE VEA COMO TU REFERENCIA
# =========================================================

type_to_depth = {tipo: i for i, tipo in enumerate(type_order)}
summary_df["depth"] = summary_df["type_1"].map(type_to_depth)

# Ajusta estos dos valores si quieres más sensación de 3D
DEPTH_X = 0.09
DEPTH_Y = -18

# X queda principalmente por generación
summary_df["x_plot"] = summary_df["generation"] + summary_df["depth"] * DEPTH_X

# Y combina altura promedio + profundidad
summary_df["y_plot"] = summary_df["avg_stats"] + summary_df["depth"] * DEPTH_Y

# =========================================================
# 9. COLORES PARA LEYENDA
# =========================================================

type_colors = {
    "Fire": "#e76f51",
    "Water": "#3a86ff",
    "Grass": "#43aa8b",
    "Electric": "#f4c430",
    "Psychic": "#d16ba5",
    "Dragon": "#6a4c93",
    "Normal": "#8d99ae",
    "Ghost": "#5a189a",
    "Fighting": "#9d0208",
    "Ice": "#4cc9f0"
}

# =========================================================
# 10. CREAR FIGURA
# =========================================================

fig = go.Figure()

# ---------------------------------------------------------
# Líneas verticales de generación
# ---------------------------------------------------------
ymin = summary_df["y_plot"].min() - 20
ymax = summary_df["y_plot"].max() + 20

for gen in range(1, 8):
    fig.add_trace(
        go.Scatter(
            x=[gen, gen + (len(type_order)-1) * DEPTH_X],
            y=[ymax, ymin],
            mode="lines",
            line=dict(color="rgba(180,180,180,0.35)", width=1),
            hoverinfo="skip",
            showlegend=False
        )
    )

# ---------------------------------------------------------
# Líneas horizontales por tipo
# ---------------------------------------------------------
for tipo in type_order:
    temp = summary_df[summary_df["type_1"] == tipo].sort_values("generation").copy()
    if len(temp) == 0:
        continue

    fig.add_trace(
        go.Scatter(
            x=temp["x_plot"],
            y=temp["y_plot"],
            mode="lines",
            line=dict(color="rgba(200,200,200,0.30)", width=1),
            hoverinfo="skip",
            showlegend=False
        )
    )

# ---------------------------------------------------------
# Puntos invisibles para hover
# ---------------------------------------------------------
fig.add_trace(
    go.Scatter(
        x=summary_df["x_plot"],
        y=summary_df["y_plot"],
        mode="markers",
        marker=dict(size=40, color="rgba(0,0,0,0)"),
        customdata=np.stack(
            [
                summary_df["pokemon"],
                summary_df["type_1"],
                summary_df["generation"],
                summary_df["avg_stats"].round(2)
            ],
            axis=-1
        ),
        hovertemplate=
            "<b>Pokémon:</b> %{customdata[0]}<br>" +
            "<b>Tipo:</b> %{customdata[1]}<br>" +
            "<b>Generación:</b> %{customdata[2]}<br>" +
            "<b>Promedio de estadísticas:</b> %{customdata[3]}<br>" +
            "<extra></extra>",
        showlegend=False
    )
)

# ---------------------------------------------------------
# Sprites
# ---------------------------------------------------------
for _, row in summary_df.iterrows():
    fig.add_layout_image(
        dict(
            source=row["sprite_base64"],
            x=row["x_plot"],
            y=row["y_plot"],
            xref="x",
            yref="y",
            sizex=0.30,
            sizey=26,
            xanchor="center",
            yanchor="middle",
            layer="above"
        )
    )

# ---------------------------------------------------------
# Leyenda
# ---------------------------------------------------------
for tipo in type_order:
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=10, color=type_colors[tipo]),
            name=tipo
        )
    )

# =========================================================
# 11. ANOTACIONES DEL EJE TIPO
# =========================================================

annotations = []

for tipo in type_order:
    d = type_to_depth[tipo]

    temp = summary_df[summary_df["type_1"] == tipo]
    if len(temp) == 0:
        continue

    y_label = temp["y_plot"].mean()
    x_label = 0.55 + d * DEPTH_X

    annotations.append(
        dict(
            x=x_label,
            y=y_label,
            text=tipo,
            showarrow=False,
            xanchor="right",
            font=dict(size=11, color="black")
        )
    )

annotations.append(
    dict(
        x=0.22,
        y=summary_df["y_plot"].min() - 10,
        text="<b>Tipo</b>",
        showarrow=False,
        font=dict(size=14, color="black")
    )
)

# =========================================================
# 12. LAYOUT FINAL
# =========================================================

fig.update_layout(
    title="<b>Scatter 3D de Pokémon: Promedio de Estadísticas por Tipo y Generación</b>",
    width=1200,
    height=760,
    plot_bgcolor="white",
    paper_bgcolor="white",
    annotations=annotations,
    margin=dict(l=90, r=220, t=80, b=90),
    legend=dict(
        title="Tipo inicial / símbolo",
        x=1.02,
        y=0.90,
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="lightgray",
        borderwidth=1
    ),
    xaxis=dict(
        title="Generación",
        tickmode="array",
        tickvals=[1, 2, 3, 4, 5, 6, 7],
        ticktext=["1", "2", "3", "4", "5", "6", "7"],
        range=[0.2, 7.9],
        showgrid=False,
        zeroline=False
    ),
    yaxis=dict(
        title="Promedio de estadísticas",
        range=[summary_df["y_plot"].min() - 30, summary_df["y_plot"].max() + 40],
        showgrid=True,
        gridcolor="rgba(210,210,210,0.35)",
        zeroline=False
    )
)

fig.show()

In [ ]:
%pip install pandas numpy matplotlib pillow requests ipympl

In [ ]:
%matplotlib widget

In [ ]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from io import BytesIO
from PIL import Image
from functools import lru_cache

from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import proj3d

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

# Hasta generación 8 son los primeros 898 Pokémon
# Gen 1: 1-151
# Gen 2: 152-251
# Gen 3: 252-386
# Gen 4: 387-493
# Gen 5: 494-649
# Gen 6: 650-721
# Gen 7: 722-809
# Gen 8: 810-898

POKEMON_INICIO = 1
POKEMON_FIN = 898

ARCHIVO_CACHE = "pokemon_pokeapi_cache.csv"

TIPOS_ORDEN = [
    "Fire",
    "Water",
    "Grass",
    "Electric",
    "Psychic",
    "Dragon",
    "Normal",
    "Ghost",
    "Fighting",
    "Ice"
]

GENERACION_MAP = {
    "generation-i": 1,
    "generation-ii": 2,
    "generation-iii": 3,
    "generation-iv": 4,
    "generation-v": 5,
    "generation-vi": 6,
    "generation-vii": 7,
    "generation-viii": 8,
    "generation-ix": 9
}


def nombre_limpio(nombre):
    return nombre.replace("-", " ").title()


def descargar_pokemon_desde_pokeapi(inicio=1, fin=898):
    datos = []

    for pokemon_id in range(inicio, fin + 1):
        try:
            url_pokemon = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
            url_species = f"https://pokeapi.co/api/v2/pokemon-species/{pokemon_id}"

            r_pokemon = requests.get(url_pokemon, timeout=20)
            r_species = requests.get(url_species, timeout=20)

            r_pokemon.raise_for_status()
            r_species.raise_for_status()

            pokemon_data = r_pokemon.json()
            species_data = r_species.json()

            nombre = nombre_limpio(pokemon_data["name"])

            tipos = pokemon_data["types"]
            tipo_principal = tipos[0]["type"]["name"].title()

            total_stats = sum(stat["base_stat"] for stat in pokemon_data["stats"])

            generacion_nombre = species_data["generation"]["name"]
            generacion = GENERACION_MAP.get(generacion_nombre, None)

            sprite = f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{pokemon_id}.png"

            datos.append({
                "ID": pokemon_id,
                "Pokemon": nombre,
                "Tipo": tipo_principal,
                "Generación": generacion,
                "Total Estadísticas": total_stats,
                "Sprite": sprite
            })

            print(f"Descargado {pokemon_id}: {nombre}")

            # Pequeña pausa para no saturar
            time.sleep(0.03)

        except Exception as e:
            print(f"Error con Pokémon {pokemon_id}: {e}")

    return pd.DataFrame(datos)


# ============================================================
# USAR CACHE SI YA EXISTE
# ============================================================

if os.path.exists(ARCHIVO_CACHE):
    print("Usando archivo cache existente...")
    df_pokemon = pd.read_csv(ARCHIVO_CACHE)
else:
    print("Descargando datos desde PokéAPI...")
    df_pokemon = descargar_pokemon_desde_pokeapi(POKEMON_INICIO, POKEMON_FIN)
    df_pokemon.to_csv(ARCHIVO_CACHE, index=False, encoding="utf-8-sig")
    print("Archivo cache creado:", ARCHIVO_CACHE)

df_pokemon.head()

In [ ]:
# ============================================================
# FILTRAR DATOS
# ============================================================

df_plot = df_pokemon.copy()

df_plot["Tipo"] = df_plot["Tipo"].astype(str).str.strip().str.title()
df_plot["Generación"] = pd.to_numeric(df_plot["Generación"], errors="coerce")
df_plot["Total Estadísticas"] = pd.to_numeric(df_plot["Total Estadísticas"], errors="coerce")

df_plot = df_plot.dropna(subset=["Pokemon", "Tipo", "Generación", "Total Estadísticas", "Sprite"])

df_plot["Generación"] = df_plot["Generación"].astype(int)

# Solo tipos de la imagen
df_plot = df_plot[df_plot["Tipo"].isin(TIPOS_ORDEN)].copy()

# Solo generaciones 1 a 8
df_plot = df_plot[df_plot["Generación"].between(1, 8)].copy()


# ============================================================
# PARA QUE SE VEA COMO LA REFERENCIA:
# 1 Pokémon por cada Tipo y Generación
# Se elige el Pokémon con mayor total de estadísticas.
# ============================================================

df_plot = (
    df_plot
    .sort_values("Total Estadísticas", ascending=False)
    .drop_duplicates(subset=["Tipo", "Generación"], keep="first")
    .copy()
)

# Crear posición Y según tipo
mapa_tipo = {tipo: i for i, tipo in enumerate(TIPOS_ORDEN)}
df_plot["y_tipo"] = df_plot["Tipo"].map(mapa_tipo)

df_plot = df_plot.sort_values(["y_tipo", "Generación"]).reset_index(drop=True)

print("Pokémon que aparecerán en el gráfico:", len(df_plot))
df_plot.head(20)

In [ ]:
@lru_cache(maxsize=1000)
def cargar_sprite(url):
    try:
        respuesta = requests.get(url, timeout=20)
        respuesta.raise_for_status()

        img = Image.open(BytesIO(respuesta.content)).convert("RGBA")

        # Aumentar nitidez visual sin deformar
        img = img.resize((96, 96), Image.Resampling.NEAREST)

        return img

    except Exception as e:
        print("No se pudo cargar sprite:", url)
        return None

In [ ]:
# ============================================================
# COLORES POR TIPO
# ============================================================

colores = {
    "Fire": "#ff5a1f",
    "Water": "#2e86de",
    "Grass": "#44aa44",
    "Electric": "#ffd700",
    "Psychic": "#f06292",
    "Dragon": "#673ab7",
    "Normal": "#b8a382",
    "Ghost": "#5e35b1",
    "Fighting": "#b03a2e",
    "Ice": "#4fc3d9"
}


# ============================================================
# CREAR FIGURA 3D
# ============================================================

fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")

x = df_plot["Generación"].values
y = df_plot["y_tipo"].values
z = df_plot["Total Estadísticas"].values

# Puntos invisibles para que el gráfico calcule bien el espacio 3D
ax.scatter(
    x,
    y,
    z,
    s=15,
    alpha=0.01,
    c=[colores[tipo] for tipo in df_plot["Tipo"]]
)


# ============================================================
# TÍTULO Y EJES
# ============================================================

ax.set_title(
    "Scatter 3D de Pokémon: Promedio de Estadísticas por Tipo y Generación",
    fontsize=16,
    fontweight="bold",
    pad=25
)

ax.set_xlabel("Generación", fontsize=12, labelpad=14)
ax.set_ylabel("Tipo", fontsize=12, labelpad=14)
ax.set_zlabel("Promedio de estadísticas", fontsize=12, labelpad=14)


# ============================================================
# CONFIGURAR EJE X
# ============================================================

ax.set_xlim(0.5, 8.5)
ax.set_xticks(range(1, 9))
ax.set_xticklabels(range(1, 9))


# ============================================================
# CONFIGURAR EJE Y
# ============================================================

ax.set_ylim(len(TIPOS_ORDEN) - 0.5, -0.5)
ax.set_yticks(range(len(TIPOS_ORDEN)))
ax.set_yticklabels(TIPOS_ORDEN)


# ============================================================
# CONFIGURAR EJE Z
# ============================================================

z_min = int(np.floor(df_plot["Total Estadísticas"].min() / 50) * 50)
z_max = int(np.ceil(df_plot["Total Estadísticas"].max() / 50) * 50)

ax.set_zlim(z_min - 30, z_max + 30)


# ============================================================
# VISTA 3D PARECIDA A TU IMAGEN
# ============================================================

ax.view_init(elev=22, azim=-62)

# Proporción de la caja 3D
ax.set_box_aspect((8, 6, 4.5))

# Rejilla
ax.grid(True)

# Fondo suave de los paneles
try:
    ax.xaxis.pane.set_alpha(0.18)
    ax.yaxis.pane.set_alpha(0.18)
    ax.zaxis.pane.set_alpha(0.18)
except:
    pass

In [ ]:
# ============================================================
# LEYENDA
# ============================================================

legend_handles = []

for tipo in TIPOS_ORDEN:
    if tipo in df_plot["Tipo"].values:
        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                label=tipo,
                markerfacecolor=colores[tipo],
                markersize=10
            )
        )

ax.legend(
    handles=legend_handles,
    title="Tipo",
    loc="upper left",
    bbox_to_anchor=(1.03, 0.95)
)

plt.tight_layout()
plt.show()

In [ ]:
%pip install pandas numpy matplotlib pillow requests

In [ ]:
%matplotlib inline

import os
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from io import BytesIO
from PIL import Image
from functools import lru_cache
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import proj3d


# ============================================================
# CONFIGURACIÓN
# ============================================================

ARCHIVO_CACHE = "pokemon_pokeapi_cache.csv"
CARPETA_SPRITES = "sprites_pokemon"
SALIDA_IMAGEN = "pokemon_3d_sprites_resultado.png"

os.makedirs(CARPETA_SPRITES, exist_ok=True)

TIPOS_ORDEN = [
    "Fire",
    "Water",
    "Grass",
    "Electric",
    "Psychic",
    "Dragon",
    "Normal",
    "Ghost",
    "Fighting",
    "Ice"
]

COLORES = {
    "Fire": "#ff5a1f",
    "Water": "#2e86de",
    "Grass": "#44aa44",
    "Electric": "#ffd700",
    "Psychic": "#f06292",
    "Dragon": "#673ab7",
    "Normal": "#b8a382",
    "Ghost": "#5e35b1",
    "Fighting": "#b03a2e",
    "Ice": "#4fc3d9"
}


# ============================================================
# FUNCIÓN PARA GENERACIÓN SEGÚN ID
# ============================================================

def obtener_generacion(pokemon_id):
    if 1 <= pokemon_id <= 151:
        return 1
    elif 152 <= pokemon_id <= 251:
        return 2
    elif 252 <= pokemon_id <= 386:
        return 3
    elif 387 <= pokemon_id <= 493:
        return 4
    elif 494 <= pokemon_id <= 649:
        return 5
    elif 650 <= pokemon_id <= 721:
        return 6
    elif 722 <= pokemon_id <= 809:
        return 7
    elif 810 <= pokemon_id <= 898:
        return 8
    else:
        return None


def limpiar_nombre(nombre):
    return nombre.replace("-", " ").title()


# ============================================================
# DESCARGAR DATOS DE POKÉAPI
# ============================================================

def descargar_datos_pokemon():
    datos = []

    for pokemon_id in range(1, 899):
        try:
            url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
            respuesta = requests.get(url, timeout=20)
            respuesta.raise_for_status()

            info = respuesta.json()

            nombre = limpiar_nombre(info["name"])
            tipo = info["types"][0]["type"]["name"].title()
            total_stats = sum(stat["base_stat"] for stat in info["stats"])
            generacion = obtener_generacion(pokemon_id)

            sprite = info["sprites"]["front_default"]

            if sprite is None:
                sprite = f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{pokemon_id}.png"

            datos.append({
                "ID": pokemon_id,
                "Pokemon": nombre,
                "Tipo": tipo,
                "Generación": generacion,
                "Total Estadísticas": total_stats,
                "Sprite": sprite
            })

            print(f"Descargado {pokemon_id}: {nombre}")

            time.sleep(0.02)

        except Exception as e:
            print(f"Error con Pokémon {pokemon_id}: {e}")

    return pd.DataFrame(datos)


# ============================================================
# CARGAR CACHE O DESCARGAR
# ============================================================

def cache_valido(ruta):
    if not os.path.exists(ruta):
        return False

    try:
        df_temp = pd.read_csv(ruta)

        columnas = [
            "ID",
            "Pokemon",
            "Tipo",
            "Generación",
            "Total Estadísticas",
            "Sprite"
        ]

        for col in columnas:
            if col not in df_temp.columns:
                return False

        if len(df_temp) < 850:
            return False

        if df_temp["Generación"].nunique() < 8:
            return False

        return True

    except:
        return False


if cache_valido(ARCHIVO_CACHE):
    print("Usando cache existente:", ARCHIVO_CACHE)
    df_pokemon = pd.read_csv(ARCHIVO_CACHE)
else:
    print("Cache no existe o está incompleto. Descargando datos...")
    df_pokemon = descargar_datos_pokemon()
    df_pokemon.to_csv(ARCHIVO_CACHE, index=False, encoding="utf-8-sig")
    print("Cache creado:", ARCHIVO_CACHE)


# ============================================================
# PREPARAR DATOS
# ============================================================

df_pokemon["Tipo"] = df_pokemon["Tipo"].astype(str).str.strip().str.title()
df_pokemon["Generación"] = pd.to_numeric(df_pokemon["Generación"], errors="coerce")
df_pokemon["Total Estadísticas"] = pd.to_numeric(df_pokemon["Total Estadísticas"], errors="coerce")
df_pokemon["ID"] = pd.to_numeric(df_pokemon["ID"], errors="coerce")

df_pokemon = df_pokemon.dropna(
    subset=["ID", "Pokemon", "Tipo", "Generación", "Total Estadísticas", "Sprite"]
)

df_pokemon["ID"] = df_pokemon["ID"].astype(int)
df_pokemon["Generación"] = df_pokemon["Generación"].astype(int)

df_pokemon = df_pokemon[
    df_pokemon["Tipo"].isin(TIPOS_ORDEN)
    & df_pokemon["Generación"].between(1, 8)
].copy()


# ============================================================
# ELEGIR UN POKÉMON POR TIPO Y GENERACIÓN
# Esto hace que se vea ordenado como la imagen.
# Toma el Pokémon con mayor total de estadísticas.
# ============================================================

df_plot = (
    df_pokemon
    .sort_values("Total Estadísticas", ascending=False)
    .drop_duplicates(subset=["Tipo", "Generación"], keep="first")
    .copy()
)

mapa_tipo = {tipo: i for i, tipo in enumerate(TIPOS_ORDEN)}
df_plot["y_tipo"] = df_plot["Tipo"].map(mapa_tipo)

df_plot = df_plot.sort_values(["y_tipo", "Generación"]).reset_index(drop=True)

print("Cantidad de Pokémon en el gráfico:", len(df_plot))
display(df_plot[["Pokemon", "Tipo", "Generación", "Total Estadísticas"]])


# ============================================================
# DESCARGAR Y CARGAR SPRITES
# ============================================================

def descargar_sprite_local(pokemon_id, url):
    ruta_local = os.path.join(CARPETA_SPRITES, f"{pokemon_id}.png")

    if os.path.exists(ruta_local):
        return ruta_local

    try:
        respuesta = requests.get(url, timeout=20)
        respuesta.raise_for_status()

        with open(ruta_local, "wb") as archivo:
            archivo.write(respuesta.content)

        return ruta_local

    except Exception as e:
        print(f"No se pudo descargar sprite de {pokemon_id}: {e}")
        return None


@lru_cache(maxsize=1000)
def cargar_sprite(pokemon_id, url):
    ruta_local = descargar_sprite_local(pokemon_id, url)

    if ruta_local is None:
        return None

    try:
        img = Image.open(ruta_local).convert("RGBA")

        # Recortar espacio transparente
        bbox = img.getbbox()
        if bbox:
            img = img.crop(bbox)

        # Tamaño fijo tipo sprite
        img = img.resize((72, 72), Image.Resampling.NEAREST)

        return img

    except Exception as e:
        print(f"No se pudo abrir sprite {pokemon_id}: {e}")
        return None


# ============================================================
# CREAR GRÁFICO 3D
# ============================================================

plt.rcParams["figure.dpi"] = 120

fig = plt.figure(figsize=(17, 10))
ax = fig.add_subplot(111, projection="3d")

# Posición del gráfico para que no se vaya a la derecha
ax.set_position([0.04, 0.08, 0.72, 0.82])

x = df_plot["Generación"].values
y = df_plot["y_tipo"].values
z = df_plot["Total Estadísticas"].values

# Puntos base visibles, por si algún sprite falla
ax.scatter(
    x,
    y,
    z,
    s=90,
    alpha=0.18,
    c=[COLORES[tipo] for tipo in df_plot["Tipo"]],
    depthshade=True
)

ax.set_title(
    "Scatter 3D de Pokémon: Promedio de Estadísticas por Tipo y Generación",
    fontsize=15,
    fontweight="bold",
    pad=25
)

ax.set_xlabel("Generación", fontsize=12, labelpad=14)
ax.set_ylabel("Tipo", fontsize=12, labelpad=14)
ax.set_zlabel("Promedio de estadísticas", fontsize=12, labelpad=14)

ax.set_xlim(0.5, 8.5)
ax.set_xticks(range(1, 9))
ax.set_xticklabels(range(1, 9))

ax.set_ylim(len(TIPOS_ORDEN) - 0.5, -0.5)
ax.set_yticks(range(len(TIPOS_ORDEN)))
ax.set_yticklabels(TIPOS_ORDEN)

z_min = int(np.floor(df_plot["Total Estadísticas"].min() / 50) * 50)
z_max = int(np.ceil(df_plot["Total Estadísticas"].max() / 50) * 50)
ax.set_zlim(z_min - 40, z_max + 40)

# Vista parecida al ejemplo
ax.view_init(elev=22, azim=-62)
ax.set_box_aspect((8, 6, 4.6))
ax.grid(True)

try:
    ax.xaxis.pane.set_alpha(0.12)
    ax.yaxis.pane.set_alpha(0.12)
    ax.zaxis.pane.set_alpha(0.12)
except:
    pass


# ============================================================
# LEYENDA
# ============================================================

legend_handles = []

for tipo in TIPOS_ORDEN:
    if tipo in df_plot["Tipo"].values:
        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                label=tipo,
                markerfacecolor=COLORES[tipo],
                markersize=10
            )
        )

ax.legend(
    handles=legend_handles,
    title="Tipo",
    loc="upper left",
    bbox_to_anchor=(1.05, 0.95)
)


# ============================================================
# DIBUJAR SPRITES COMO PUNTOS 3D
# Esta parte está corregida para VS Code.
# ============================================================

fig.canvas.draw()

for _, fila in df_plot.iterrows():
    img = cargar_sprite(int(fila["ID"]), fila["Sprite"])

    if img is None:
        continue

    x3 = fila["Generación"]
    y3 = fila["y_tipo"]
    z3 = fila["Total Estadísticas"]

    # Convertir coordenada 3D a coordenada 2D de pantalla
    x2, y2, z2 = proj3d.proj_transform(x3, y3, z3, ax.get_proj())

    # Convertir a pixeles de la figura
    xpix, ypix = ax.transData.transform((x2, y2))

    imagebox = OffsetImage(img, zoom=0.55, resample=True)

    ab = AnnotationBbox(
        imagebox,
        (xpix, ypix),
        xycoords="figure pixels",
        frameon=False,
        pad=0.0,
        box_alignment=(0.5, 0.5),
        annotation_clip=False
    )

    ab.set_zorder(int(10000 - z2 * 1000))
    fig.add_artist(ab)


# ============================================================
# GUARDAR Y MOSTRAR
# ============================================================

fig.savefig(SALIDA_IMAGEN, dpi=160)
print("Imagen guardada como:", SALIDA_IMAGEN)

plt.show()

In [ ]:
%pip install pandas numpy requests pillow plotly nbformat

In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from PIL import Image
from functools import lru_cache


# Para VS Code Notebook
pio.renderers.default = "vscode"


# ============================================================
# CONFIGURACIÓN
# ============================================================

ARCHIVO_CACHE = "pokemon_pokeapi_cache.csv"
CARPETA_SPRITES = "sprites_pokemon"

os.makedirs(CARPETA_SPRITES, exist_ok=True)

# Cambia este número para mostrar más Pokémon
# 1 = pocos
# 3 = recomendado
# 5 = más lleno
# 10 = muchos, puede tardar más
TOP_N_POR_TIPO_GENERACION = 3

# Si se tarda mucho, baja a 16 o 14
PIXEL_SPRITE = 18

TIPOS_ORDEN = [
    "Fire",
    "Water",
    "Grass",
    "Electric",
    "Psychic",
    "Dragon",
    "Normal",
    "Ghost",
    "Fighting",
    "Ice"
]

COLORES = {
    "Fire": "#ff5a1f",
    "Water": "#2e86de",
    "Grass": "#44aa44",
    "Electric": "#ffd700",
    "Psychic": "#f06292",
    "Dragon": "#673ab7",
    "Normal": "#b8a382",
    "Ghost": "#5e35b1",
    "Fighting": "#b03a2e",
    "Ice": "#4fc3d9"
}


# ============================================================
# GENERACIÓN SEGÚN ID
# ============================================================

def obtener_generacion(pokemon_id):
    if 1 <= pokemon_id <= 151:
        return 1
    elif 152 <= pokemon_id <= 251:
        return 2
    elif 252 <= pokemon_id <= 386:
        return 3
    elif 387 <= pokemon_id <= 493:
        return 4
    elif 494 <= pokemon_id <= 649:
        return 5
    elif 650 <= pokemon_id <= 721:
        return 6
    elif 722 <= pokemon_id <= 809:
        return 7
    elif 810 <= pokemon_id <= 898:
        return 8
    return None


def limpiar_nombre(nombre):
    return nombre.replace("-", " ").title()


# ============================================================
# DESCARGAR DATOS DESDE POKÉAPI
# ============================================================

def descargar_datos_pokemon():
    datos = []

    for pokemon_id in range(1, 899):
        try:
            url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
            respuesta = requests.get(url, timeout=20)
            respuesta.raise_for_status()

            info = respuesta.json()

            nombre = limpiar_nombre(info["name"])
            tipo = info["types"][0]["type"]["name"].title()
            total_stats = sum(stat["base_stat"] for stat in info["stats"])
            generacion = obtener_generacion(pokemon_id)

            sprite = info["sprites"]["front_default"]

            if sprite is None:
                sprite = f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{pokemon_id}.png"

            datos.append({
                "ID": pokemon_id,
                "Pokemon": nombre,
                "Tipo": tipo,
                "Generación": generacion,
                "Total Estadísticas": total_stats,
                "Sprite": sprite
            })

            print(f"Descargado {pokemon_id}: {nombre}")
            time.sleep(0.02)

        except Exception as e:
            print(f"Error con Pokémon {pokemon_id}: {e}")

    return pd.DataFrame(datos)


def cache_valido(ruta):
    if not os.path.exists(ruta):
        return False

    try:
        df_temp = pd.read_csv(ruta)

        columnas = [
            "ID",
            "Pokemon",
            "Tipo",
            "Generación",
            "Total Estadísticas",
            "Sprite"
        ]

        for col in columnas:
            if col not in df_temp.columns:
                return False

        return len(df_temp) >= 850

    except:
        return False


if cache_valido(ARCHIVO_CACHE):
    print("Usando cache existente:", ARCHIVO_CACHE)
    df_pokemon = pd.read_csv(ARCHIVO_CACHE)
else:
    print("Descargando datos desde PokéAPI...")
    df_pokemon = descargar_datos_pokemon()
    df_pokemon.to_csv(ARCHIVO_CACHE, index=False, encoding="utf-8-sig")
    print("Cache creado:", ARCHIVO_CACHE)


# ============================================================
# PREPARAR DATOS
# ============================================================

df_pokemon["Tipo"] = df_pokemon["Tipo"].astype(str).str.strip().str.title()
df_pokemon["Generación"] = pd.to_numeric(df_pokemon["Generación"], errors="coerce")
df_pokemon["Total Estadísticas"] = pd.to_numeric(df_pokemon["Total Estadísticas"], errors="coerce")
df_pokemon["ID"] = pd.to_numeric(df_pokemon["ID"], errors="coerce")

df_pokemon = df_pokemon.dropna(
    subset=["ID", "Pokemon", "Tipo", "Generación", "Total Estadísticas", "Sprite"]
)

df_pokemon["ID"] = df_pokemon["ID"].astype(int)
df_pokemon["Generación"] = df_pokemon["Generación"].astype(int)

df_pokemon = df_pokemon[
    df_pokemon["Tipo"].isin(TIPOS_ORDEN)
    & df_pokemon["Generación"].between(1, 8)
].copy()


# ============================================================
# ELEGIR MÁS POKÉMON POR TIPO Y GENERACIÓN
# ============================================================

df_plot = (
    df_pokemon
    .sort_values(
        ["Tipo", "Generación", "Total Estadísticas"],
        ascending=[True, True, False]
    )
    .groupby(["Tipo", "Generación"], group_keys=False)
    .head(TOP_N_POR_TIPO_GENERACION)
    .copy()
)

mapa_tipo = {tipo: i for i, tipo in enumerate(TIPOS_ORDEN)}
df_plot["tipo_index"] = df_plot["Tipo"].map(mapa_tipo)

df_plot = df_plot.sort_values(
    ["tipo_index", "Generación", "Total Estadísticas"],
    ascending=[True, True, False]
).reset_index(drop=True)


# ============================================================
# OFFSETS PARA QUE NO SE ENCIMEN
# ============================================================

df_plot["pos_en_celda"] = df_plot.groupby(["Tipo", "Generación"]).cumcount()

offsets = [
    (0.00, 0.00),
    (-0.23, -0.23),
    (0.23, 0.23),
    (-0.23, 0.23),
    (0.23, -0.23),
    (0.00, -0.33),
    (0.00, 0.33),
    (-0.33, 0.00),
    (0.33, 0.00),
    (-0.36, -0.36),
    (0.36, 0.36),
    (-0.36, 0.36),
    (0.36, -0.36),
]

df_plot["offset_x"] = df_plot["pos_en_celda"].apply(
    lambda i: offsets[i % len(offsets)][0]
)

df_plot["offset_y"] = df_plot["pos_en_celda"].apply(
    lambda i: offsets[i % len(offsets)][1]
)

df_plot["x_plot"] = df_plot["Generación"] + df_plot["offset_x"]
df_plot["y_plot"] = df_plot["tipo_index"] + df_plot["offset_y"]

print("Cantidad de Pokémon en el gráfico:", len(df_plot))
display(df_plot[["Pokemon", "Tipo", "Generación", "Total Estadísticas"]])


# ============================================================
# DESCARGAR SPRITES
# ============================================================

def descargar_sprite_local(pokemon_id, url):
    ruta_local = os.path.join(CARPETA_SPRITES, f"{pokemon_id}.png")

    if os.path.exists(ruta_local):
        return ruta_local

    try:
        respuesta = requests.get(url, timeout=20)
        respuesta.raise_for_status()

        with open(ruta_local, "wb") as archivo:
            archivo.write(respuesta.content)

        return ruta_local

    except Exception as e:
        print(f"No se pudo descargar sprite de {pokemon_id}: {e}")
        return None


@lru_cache(maxsize=1000)
def cargar_sprite_array(pokemon_id, url, size=18):
    ruta = descargar_sprite_local(int(pokemon_id), url)

    if ruta is None:
        return None

    try:
        img = Image.open(ruta).convert("RGBA")

        bbox = img.getbbox()
        if bbox:
            img = img.crop(bbox)

        img = img.resize((size, size), Image.Resampling.NEAREST)

        return np.array(img)

    except Exception as e:
        print(f"No se pudo procesar sprite {pokemon_id}: {e}")
        return None


# ============================================================
# CONVERTIR CADA SPRITE A MALLA 3D
# ============================================================

SPRITE_ANCHO_X = 0.55
SPRITE_ALTO_Z = 65

xs = []
ys = []
zs = []

ii = []
jj = []
kk = []

facecolors = []

indice = 0

for _, fila in df_plot.iterrows():
    arr = cargar_sprite_array(
        int(fila["ID"]),
        fila["Sprite"],
        size=PIXEL_SPRITE
    )

    if arr is None:
        continue

    centro_x = float(fila["x_plot"])
    centro_y = float(fila["y_plot"])
    centro_z = float(fila["Total Estadísticas"])

    alto, ancho, _ = arr.shape

    pixel_w = SPRITE_ANCHO_X / ancho
    pixel_h = SPRITE_ALTO_Z / alto

    for py in range(alto):
        for px in range(ancho):
            r, g, b, a = arr[py, px]

            if a < 40:
                continue

            x0 = centro_x - SPRITE_ANCHO_X / 2 + px * pixel_w
            x1 = x0 + pixel_w

            z1 = centro_z + SPRITE_ALTO_Z / 2 - py * pixel_h
            z0 = z1 - pixel_h

            y = centro_y

            xs.extend([x0, x1, x1, x0])
            ys.extend([y, y, y, y])
            zs.extend([z0, z0, z1, z1])

            ii.extend([indice, indice])
            jj.extend([indice + 1, indice + 2])
            kk.extend([indice + 2, indice + 3])

            color = f"rgba({r},{g},{b},{a / 255})"
            facecolors.extend([color, color])

            indice += 4


mesh_sprites = go.Mesh3d(
    x=xs,
    y=ys,
    z=zs,
    i=ii,
    j=jj,
    k=kk,
    facecolor=facecolors,
    flatshading=True,
    hoverinfo="skip",
    lighting=dict(
        ambient=1,
        diffuse=0,
        specular=0,
        roughness=1,
        fresnel=0
    ),
    showscale=False,
    name="Sprites Pokémon"
)


# ============================================================
# CREAR FIGURA
# ============================================================

fig = go.Figure()

fig.add_trace(mesh_sprites)


# ============================================================
# PUNTOS INVISIBLES PARA HOVER CARD
# ============================================================

for tipo in TIPOS_ORDEN:
    data_tipo = df_plot[df_plot["Tipo"] == tipo].copy()

    if data_tipo.empty:
        continue

    customdata = np.stack(
        [
            data_tipo["Pokemon"],
            data_tipo["Tipo"],
            data_tipo["ID"],
            data_tipo["Total Estadísticas"],
            data_tipo["Generación"]
        ],
        axis=-1
    )

    fig.add_trace(
        go.Scatter3d(
            x=data_tipo["x_plot"],
            y=data_tipo["y_plot"],
            z=data_tipo["Total Estadísticas"],
            mode="markers",
            name=tipo,
            customdata=customdata,
            marker=dict(
                size=9,
                color=COLORES[tipo],
                opacity=0.13
            ),
            hovertemplate=
                "<b style='font-size:15px'>%{customdata[0]}</b><br><br>" +
                "<b>Tipo:</b> %{customdata[1]}<br>" +
                "<b>Generación:</b> %{customdata[4]}<br>" +
                "<b>Total de estadísticas:</b> %{customdata[3]}<br>" +
                "<b>ID Pokédex:</b> %{customdata[2]}<br>" +
                "<br><b>Coordenadas</b><br>" +
                "X = Generación %{customdata[4]}<br>" +
                "Y = %{customdata[1]}<br>" +
                "Z = %{customdata[3]}" +
                "<extra></extra>"
        )
    )


# ============================================================
# CONFIGURAR ESCENA 3D
# ============================================================

z_min = int(np.floor(df_plot["Total Estadísticas"].min() / 50) * 50) - 50
z_max = int(np.ceil(df_plot["Total Estadísticas"].max() / 50) * 50) + 50

fig.update_layout(
    title=dict(
        text=f"Scatter 3D de Pokémon: Top {TOP_N_POR_TIPO_GENERACION} por Tipo y Generación",
        x=0.02,
        y=0.95,
        font=dict(size=22, color="#22345b")
    ),

    width=1250,
    height=820,

    margin=dict(l=0, r=0, t=70, b=0),

    legend=dict(
        title="Tipo",
        x=0.86,
        y=0.85,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#d9e1ea",
        borderwidth=1
    ),

    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#cbd5e1",
        font=dict(size=13, color="#22345b")
    ),

    scene=dict(
        xaxis=dict(
            title="Generación",
            tickmode="array",
            tickvals=list(range(1, 9)),
            ticktext=[str(i) for i in range(1, 9)],
            range=[0.4, 8.6],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False
        ),

        yaxis=dict(
            title="Tipo",
            tickmode="array",
            tickvals=list(range(len(TIPOS_ORDEN))),
            ticktext=TIPOS_ORDEN,
            range=[len(TIPOS_ORDEN) - 0.7, -0.7],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False
        ),

        zaxis=dict(
            title="Promedio de estadísticas",
            range=[z_min, z_max],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False
        ),

        camera=dict(
            eye=dict(x=1.65, y=-2.15, z=0.95)
        ),

        aspectmode="manual",
        aspectratio=dict(x=1.5, y=1.15, z=0.9)
    )
)


# ============================================================
# MOSTRAR DENTRO DEL NOTEBOOK
# ============================================================

fig.show()

In [ ]:
%pip install kagglehub pandas numpy pillow plotly nbformat requests

In [ ]:
import os
import re
import time
from pathlib import Path
from functools import lru_cache

import kagglehub
import pandas as pd
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

import requests
import plotly.graph_objects as go
import plotly.io as pio


# ============================================================
# CONFIGURACIÓN
# ============================================================

pio.renderers.default = "vscode"

DATASET_KAGGLE = "divyanshusingh369/complete-pokemon-library-32k-images-and-csv"

ARCHIVO_CACHE_POKEAPI = "pokemon_pokeapi_cache.csv"

TOP_N_POR_TIPO_GENERACION = 3

# Más resolución para que las imágenes se vean claras
PIXEL_SPRITE = 32

# Tamaño visual de cada Pokémon dentro del gráfico
SPRITE_ANCHO_X = 0.75
SPRITE_ALTO_Z = 95

# Déjalo True si las imágenes tienen fondo blanco
QUITAR_FONDO_CLARO = True

TIPOS_ORDEN = [
    "Fire",
    "Water",
    "Grass",
    "Electric",
    "Psychic",
    "Dragon",
    "Normal",
    "Ghost",
    "Fighting",
    "Ice"
]

COLORES = {
    "Fire": "#ff5a1f",
    "Water": "#2e86de",
    "Grass": "#44aa44",
    "Electric": "#ffd700",
    "Psychic": "#f06292",
    "Dragon": "#673ab7",
    "Normal": "#b8a382",
    "Ghost": "#5e35b1",
    "Fighting": "#b03a2e",
    "Ice": "#4fc3d9"
}


# ============================================================
# FUNCIONES BASE
# ============================================================

def limpiar_nombre(nombre):
    return str(nombre).replace("-", " ").replace("_", " ").strip().title()


def normalizar_nombre(nombre):
    nombre = str(nombre).lower().strip()

    reemplazos = {
        "♀": " female",
        "♂": " male",
        "mr.": "mr",
        "mime jr.": "mime jr",
        "type: null": "typenull",
        "farfetch'd": "farfetchd",
        "sirfetch'd": "sirfetchd",
    }

    for a, b in reemplazos.items():
        nombre = nombre.replace(a, b)

    nombre = nombre.replace(".", "")
    nombre = nombre.replace("'", "")
    nombre = nombre.replace(":", "")
    nombre = nombre.replace("-", " ")
    nombre = nombre.replace("_", " ")
    nombre = re.sub(r"\s+", " ", nombre)

    return re.sub(r"[^a-z0-9]+", "", nombre)


def normalizar_base_para_merge(nombre):
    nombre = str(nombre).strip()

    prefijos = [
        "Mega ",
        "Galarian ",
        "Alolan ",
        "Hisuian ",
        "Paldean ",
        "Primal ",
        "Origin ",
        "Attack Forme ",
        "Defense Forme ",
        "Speed Forme ",
        "Normal Forme ",
    ]

    for p in prefijos:
        if nombre.startswith(p):
            nombre = nombre.replace(p, "", 1)

    return normalizar_nombre(nombre)


def obtener_generacion(pokemon_id):
    if 1 <= pokemon_id <= 151:
        return 1
    elif 152 <= pokemon_id <= 251:
        return 2
    elif 252 <= pokemon_id <= 386:
        return 3
    elif 387 <= pokemon_id <= 493:
        return 4
    elif 494 <= pokemon_id <= 649:
        return 5
    elif 650 <= pokemon_id <= 721:
        return 6
    elif 722 <= pokemon_id <= 809:
        return 7
    elif 810 <= pokemon_id <= 898:
        return 8
    elif 899 <= pokemon_id <= 1025:
        return 9
    return np.nan


def extraer_tipo_principal(valor):
    texto = str(valor).replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    return texto.split(",")[0].split("/")[0].split("|")[0].strip().title()


# ============================================================
# 1. DESCARGAR DATASET DE KAGGLE
# ============================================================

print("Descargando / localizando dataset de Kaggle...")

dataset_path = Path(kagglehub.dataset_download(DATASET_KAGGLE))

print("Dataset Kaggle:")
print(dataset_path)

csv_files = list(dataset_path.rglob("*.csv"))

image_extensions = {".png", ".jpg", ".jpeg", ".webp"}
image_files = [
    p for p in dataset_path.rglob("*")
    if p.suffix.lower() in image_extensions
]

print("CSV encontrados:", len(csv_files))
print("Imágenes encontradas:", len(image_files))

if len(csv_files) == 0:
    raise ValueError("No se encontró ningún CSV en el dataset de Kaggle.")

csv_elegido = None

for f in csv_files:
    if f.name.lower() == "pokemondb_dataset.csv":
        csv_elegido = f
        break

if csv_elegido is None:
    csv_elegido = csv_files[0]

print("CSV usado:", csv_elegido.name)

df_kaggle = pd.read_csv(csv_elegido)

print("Columnas del CSV:")
print(df_kaggle.columns.tolist())


# ============================================================
# 2. PREPARAR DATOS DEL CSV DE KAGGLE
# ============================================================

columnas_necesarias = [
    "Pokemon",
    "Type",
    "HP Base",
    "Attack Base",
    "Defense Base",
    "Special Attack Base",
    "Special Defense Base",
    "Speed Base"
]

faltantes = [c for c in columnas_necesarias if c not in df_kaggle.columns]

if faltantes:
    raise ValueError(f"Faltan estas columnas en el CSV de Kaggle: {faltantes}")

df_kaggle_limpio = pd.DataFrame()

df_kaggle_limpio["Pokemon"] = df_kaggle["Pokemon"].astype(str).apply(limpiar_nombre)
df_kaggle_limpio["Pokemon_key"] = df_kaggle_limpio["Pokemon"].apply(normalizar_nombre)
df_kaggle_limpio["Pokemon_key_base"] = df_kaggle_limpio["Pokemon"].apply(normalizar_base_para_merge)

df_kaggle_limpio["Tipo"] = df_kaggle["Type"].apply(extraer_tipo_principal)

stats_cols = [
    "HP Base",
    "Attack Base",
    "Defense Base",
    "Special Attack Base",
    "Special Defense Base",
    "Speed Base"
]

df_kaggle_limpio["Total Estadísticas"] = (
    df_kaggle[stats_cols]
    .apply(pd.to_numeric, errors="coerce")
    .sum(axis=1)
)

df_kaggle_limpio = df_kaggle_limpio.dropna(
    subset=["Pokemon", "Pokemon_key", "Tipo", "Total Estadísticas"]
)

df_kaggle_limpio["Total Estadísticas"] = df_kaggle_limpio["Total Estadísticas"].astype(int)

print("Registros limpios desde Kaggle:", len(df_kaggle_limpio))


# ============================================================
# 3. OBTENER ID Y GENERACIÓN DESDE CACHE / POKÉAPI
# ============================================================

def descargar_cache_pokeapi():
    datos = []

    for pokemon_id in range(1, 899):
        try:
            url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            info = r.json()

            nombre = limpiar_nombre(info["name"])
            tipo = info["types"][0]["type"]["name"].title()
            total = sum(stat["base_stat"] for stat in info["stats"])

            datos.append({
                "ID": pokemon_id,
                "Pokemon": nombre,
                "Tipo": tipo,
                "Generación": obtener_generacion(pokemon_id),
                "Total Estadísticas": total
            })

            print(f"Descargado {pokemon_id}: {nombre}")
            time.sleep(0.02)

        except Exception as e:
            print(f"Error con Pokémon {pokemon_id}: {e}")

    return pd.DataFrame(datos)


def cache_pokeapi_valido(ruta):
    if not os.path.exists(ruta):
        return False

    try:
        temp = pd.read_csv(ruta)
        cols = temp.columns.tolist()

        return (
            "ID" in cols
            and "Pokemon" in cols
            and "Generación" in cols
            and len(temp) >= 850
        )
    except:
        return False


if cache_pokeapi_valido(ARCHIVO_CACHE_POKEAPI):
    print("Usando cache PokeAPI existente:", ARCHIVO_CACHE_POKEAPI)
    df_pokeapi = pd.read_csv(ARCHIVO_CACHE_POKEAPI)
else:
    print("No hay cache PokeAPI válido. Descargando datos...")
    df_pokeapi = descargar_cache_pokeapi()
    df_pokeapi.to_csv(ARCHIVO_CACHE_POKEAPI, index=False, encoding="utf-8-sig")

df_pokeapi["Pokemon"] = df_pokeapi["Pokemon"].astype(str).apply(limpiar_nombre)
df_pokeapi["Pokemon_key"] = df_pokeapi["Pokemon"].apply(normalizar_nombre)
df_pokeapi["Pokemon_key_base"] = df_pokeapi["Pokemon"].apply(normalizar_base_para_merge)

df_pokeapi["ID"] = pd.to_numeric(df_pokeapi["ID"], errors="coerce")
df_pokeapi["Generación"] = pd.to_numeric(df_pokeapi["Generación"], errors="coerce")

df_pokeapi = df_pokeapi.dropna(subset=["ID", "Generación"])

df_pokeapi["ID"] = df_pokeapi["ID"].astype(int)
df_pokeapi["Generación"] = df_pokeapi["Generación"].astype(int)

df_mapa = df_pokeapi[["ID", "Pokemon_key", "Pokemon_key_base", "Generación"]].copy()


# Merge exacto
df_pokemon = df_kaggle_limpio.merge(
    df_mapa[["ID", "Pokemon_key", "Generación"]],
    on="Pokemon_key",
    how="left"
)

# Merge base para los que no encontraron generación
faltan = df_pokemon["Generación"].isna()

if faltan.any():
    mapa_base = (
        df_mapa
        .drop_duplicates("Pokemon_key_base")
        [["ID", "Pokemon_key_base", "Generación"]]
    )

    df_extra = df_pokemon.loc[faltan, ["Pokemon_key_base"]].merge(
        mapa_base,
        on="Pokemon_key_base",
        how="left"
    )

    df_pokemon.loc[faltan, "ID"] = df_extra["ID"].values
    df_pokemon.loc[faltan, "Generación"] = df_extra["Generación"].values

df_pokemon = df_pokemon.dropna(subset=["ID", "Generación"])

df_pokemon["ID"] = df_pokemon["ID"].astype(int)
df_pokemon["Generación"] = df_pokemon["Generación"].astype(int)

print("Pokémon con generación encontrada:", len(df_pokemon))


# ============================================================
# 4. INDEXAR IMÁGENES DE KAGGLE
# ============================================================

def puntuar_imagen(path):
    texto = str(path).lower()
    nombre = path.name.lower()

    score = 0

    preferidas = [
        "front",
        "default",
        "normal",
        "official",
        "artwork",
        "pokemon"
    ]

    evitar = [
        "back",
        "shiny",
        "female",
        "icon",
        "small",
        "thumb",
        "generation"
    ]

    for p in preferidas:
        if p in texto:
            score += 8

    for e in evitar:
        if e in texto:
            score -= 12

    if path.suffix.lower() == ".png":
        score += 3

    score -= len(path.parts) * 0.01

    return score


imagen_por_key = {}

for path in image_files:
    keys = set()

    keys.add(normalizar_nombre(path.stem))
    keys.add(normalizar_nombre(path.parent.name))

    # También indexa carpeta padre del padre
    if path.parent.parent:
        keys.add(normalizar_nombre(path.parent.parent.name))

    for key in keys:
        if not key:
            continue

        if key not in imagen_por_key:
            imagen_por_key[key] = []

        imagen_por_key[key].append(path)

for key in imagen_por_key:
    imagen_por_key[key] = sorted(
        imagen_por_key[key],
        key=puntuar_imagen,
        reverse=True
    )


def buscar_imagen(nombre, key, key_base):
    posibles = [
        key,
        key_base,
        normalizar_nombre(nombre),
        normalizar_base_para_merge(nombre)
    ]

    for k in posibles:
        if k in imagen_por_key and len(imagen_por_key[k]) > 0:
            return imagen_por_key[k][0]

    # Búsqueda flexible
    for k in posibles:
        if not k:
            continue

        for img_key, paths in imagen_por_key.items():
            if k in img_key or img_key in k:
                if len(paths) > 0:
                    return paths[0]

    return None


df_pokemon["Imagen"] = df_pokemon.apply(
    lambda row: buscar_imagen(
        row["Pokemon"],
        row["Pokemon_key"],
        row["Pokemon_key_base"]
    ),
    axis=1
)

df_pokemon = df_pokemon.dropna(subset=["Imagen"]).copy()

print("Pokémon con imagen de Kaggle:", len(df_pokemon))

df_pokemon = df_pokemon[
    df_pokemon["Tipo"].isin(TIPOS_ORDEN)
    & df_pokemon["Generación"].between(1, 8)
].copy()

print("Pokémon después de filtrar tipo y generación:", len(df_pokemon))

if df_pokemon.empty:
    raise ValueError(
        "No quedaron Pokémon después de filtrar. "
        "Revisa TIPOS_ORDEN o el emparejamiento de nombres/imágenes."
    )


# ============================================================
# 5. SELECCIONAR TOP N POR TIPO Y GENERACIÓN
# ============================================================

df_plot = (
    df_pokemon
    .sort_values(
        ["Tipo", "Generación", "Total Estadísticas"],
        ascending=[True, True, False]
    )
    .groupby(["Tipo", "Generación"], group_keys=False)
    .head(TOP_N_POR_TIPO_GENERACION)
    .copy()
)

mapa_tipo = {tipo: i for i, tipo in enumerate(TIPOS_ORDEN)}
df_plot["tipo_index"] = df_plot["Tipo"].map(mapa_tipo)

df_plot = df_plot.sort_values(
    ["tipo_index", "Generación", "Total Estadísticas"],
    ascending=[True, True, False]
).reset_index(drop=True)

df_plot["pos_en_celda"] = df_plot.groupby(["Tipo", "Generación"]).cumcount()

offsets = [
    (0.00, 0.00),
    (-0.23, -0.23),
    (0.23, 0.23),
    (-0.23, 0.23),
    (0.23, -0.23),
    (0.00, -0.33),
    (0.00, 0.33),
    (-0.33, 0.00),
    (0.33, 0.00),
    (-0.36, -0.36),
    (0.36, 0.36),
    (-0.36, 0.36),
    (0.36, -0.36),
]

df_plot["offset_x"] = df_plot["pos_en_celda"].apply(lambda i: offsets[i % len(offsets)][0])
df_plot["offset_y"] = df_plot["pos_en_celda"].apply(lambda i: offsets[i % len(offsets)][1])

df_plot["x_plot"] = df_plot["Generación"] + df_plot["offset_x"]
df_plot["y_plot"] = df_plot["tipo_index"] + df_plot["offset_y"]

print("Cantidad de Pokémon en el gráfico:", len(df_plot))
display(df_plot[["Pokemon", "Tipo", "Generación", "Total Estadísticas", "Imagen"]])


# ============================================================
# 6. CARGAR IMÁGENES KAGGLE
# ============================================================

@lru_cache(maxsize=2000)
def cargar_imagen_kaggle(path_str, size=32):
    try:
        img = Image.open(path_str).convert("RGBA")

        # Quitar fondo blanco o casi blanco
        if QUITAR_FONDO_CLARO:
            arr = np.array(img)

            r = arr[:, :, 0]
            g = arr[:, :, 1]
            b = arr[:, :, 2]
            a = arr[:, :, 3]

            # Fondo muy claro
            fondo_claro = (r > 248) & (g > 248) & (b > 248)

            arr[:, :, 3] = np.where(fondo_claro, 0, a)

            img = Image.fromarray(arr, mode="RGBA")

        # Recortar bordes transparentes
        bbox = img.getbbox()
        if bbox:
            img = img.crop(bbox)

        # Aumentar tamaño antes de reducir para conservar mejor detalle
        img = img.resize(
            (img.width * 2, img.height * 2),
            Image.Resampling.LANCZOS
        )

        # Mejorar nitidez y contraste
        img = ImageEnhance.Sharpness(img).enhance(2.2)
        img = ImageEnhance.Contrast(img).enhance(1.15)
        img = ImageEnhance.Color(img).enhance(1.15)

        # Mantener proporción
        img.thumbnail((size, size), Image.Resampling.LANCZOS)

        # Crear lienzo transparente cuadrado
        lienzo = Image.new("RGBA", (size, size), (0, 0, 0, 0))

        x = (size - img.width) // 2
        y = (size - img.height) // 2

        lienzo.paste(img, (x, y), img)

        return np.array(lienzo)

    except Exception as e:
        print("No se pudo cargar imagen:", path_str, e)
        return None


# ============================================================
# 7. CONVERTIR IMÁGENES A MALLA 3D
# ============================================================

xs = []
ys = []
zs = []

ii = []
jj = []
kk = []

facecolors = []

indice = 0

for _, fila in df_plot.iterrows():
    arr = cargar_imagen_kaggle(str(fila["Imagen"]), size=PIXEL_SPRITE)

    if arr is None:
        continue

    centro_x = float(fila["x_plot"])
    centro_y = float(fila["y_plot"])
    centro_z = float(fila["Total Estadísticas"])

    alto, ancho, _ = arr.shape

    pixel_w = SPRITE_ANCHO_X / ancho
    pixel_h = SPRITE_ALTO_Z / alto

    for py in range(alto):
        for px in range(ancho):
            r, g, b, a = arr[py, px]

            if a < 40:
                continue

            x0 = centro_x - SPRITE_ANCHO_X / 2 + px * pixel_w
            x1 = x0 + pixel_w

            z1 = centro_z + SPRITE_ALTO_Z / 2 - py * pixel_h
            z0 = z1 - pixel_h

            y = centro_y

            xs.extend([x0, x1, x1, x0])
            ys.extend([y, y, y, y])
            zs.extend([z0, z0, z1, z1])

            ii.extend([indice, indice])
            jj.extend([indice + 1, indice + 2])
            kk.extend([indice + 2, indice + 3])

            color = f"rgba({r},{g},{b},{a / 255})"
            facecolors.extend([color, color])

            indice += 4

if len(xs) == 0:
    raise ValueError(
        "No se generaron pixeles 3D. "
        "Las imágenes no se cargaron correctamente o quedaron transparentes."
    )

mesh_sprites = go.Mesh3d(
    x=xs,
    y=ys,
    z=zs,
    i=ii,
    j=jj,
    k=kk,
    facecolor=facecolors,
    flatshading=True,
    hoverinfo="skip",
    lighting=dict(
        ambient=1,
        diffuse=0,
        specular=0,
        roughness=1,
        fresnel=0
    ),
    showscale=False,
    name="Imágenes Kaggle"
)

print("Pixeles 3D generados:", len(xs))


# ============================================================
# 8. CREAR FIGURA 3D INTERACTIVA
# ============================================================

fig = go.Figure()

fig.add_trace(mesh_sprites)

for tipo in TIPOS_ORDEN:
    data_tipo = df_plot[df_plot["Tipo"] == tipo].copy()

    if data_tipo.empty:
        continue

    customdata = np.stack(
        [
            data_tipo["Pokemon"],
            data_tipo["Tipo"],
            data_tipo["ID"],
            data_tipo["Total Estadísticas"],
            data_tipo["Generación"]
        ],
        axis=-1
    )

    fig.add_trace(
        go.Scatter3d(
            x=data_tipo["x_plot"],
            y=data_tipo["y_plot"],
            z=data_tipo["Total Estadísticas"],
            mode="markers",
            name=tipo,
            customdata=customdata,
            marker=dict(
                size=9,
                color=COLORES[tipo],
                opacity=0.13
            ),
            hovertemplate=
                "<b style='font-size:15px'>%{customdata[0]}</b><br><br>" +
                "<b>Tipo:</b> %{customdata[1]}<br>" +
                "<b>Generación:</b> %{customdata[4]}<br>" +
                "<b>Total de estadísticas:</b> %{customdata[3]}<br>" +
                "<b>ID:</b> %{customdata[2]}<br>" +
                "<br><b>Coordenadas</b><br>" +
                "X = Generación %{customdata[4]}<br>" +
                "Y = %{customdata[1]}<br>" +
                "Z = %{customdata[3]}" +
                "<extra></extra>"
        )
    )


# ============================================================
# 9. CONFIGURAR ESCENA
# ============================================================

z_min = int(np.floor(df_plot["Total Estadísticas"].min() / 50) * 50) - 50
z_max = int(np.ceil(df_plot["Total Estadísticas"].max() / 50) * 50) + 50

fig.update_layout(
    title=dict(
        text=f"Scatter 3D de Pokémon con imágenes de Kaggle: Top {TOP_N_POR_TIPO_GENERACION} por Tipo y Generación",
        x=0.02,
        y=0.95,
        font=dict(size=22, color="#22345b")
    ),

    width=1250,
    height=820,

    margin=dict(l=0, r=0, t=70, b=0),

    legend=dict(
        title="Tipo",
        x=0.86,
        y=0.85,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#d9e1ea",
        borderwidth=1
    ),

    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#cbd5e1",
        font=dict(size=13, color="#22345b")
    ),

    scene=dict(
        xaxis=dict(
            title="Generación",
            tickmode="array",
            tickvals=list(range(1, 9)),
            ticktext=[str(i) for i in range(1, 9)],
            range=[0.4, 8.6],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False
        ),

        yaxis=dict(
            title="Tipo",
            tickmode="array",
            tickvals=list(range(len(TIPOS_ORDEN))),
            ticktext=TIPOS_ORDEN,
            range=[len(TIPOS_ORDEN) - 0.7, -0.7],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False
        ),

        zaxis=dict(
            title="Promedio de estadísticas",
            range=[z_min, z_max],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False
        ),

        camera=dict(
            eye=dict(x=1.65, y=-2.15, z=0.95)
        ),

        aspectmode="manual",
        aspectratio=dict(x=1.35, y=1.10, z=1.45)
    )
)

fig.show()

# 11. Pokémon personalizados

En esta sección se agregan **11 Pokémon de creación propia** al mismo dataset y a los
gráficos 3D que ya existían.

**Requisito:** la carpeta `pokemon_personalizados/` debe estar junto a este notebook.
Contiene las imágenes ya recortadas y con fondo transparente:

| Familia | Etapas |
|---|---|
| Pochoco | Pochoco → Pochotron → Pocholord → Mega Pocholord |
| Mariposon | Mariposon → Maripopon → Maripolord → Mega Maripolord |
| Individuales | Luneytoons, Pajaroloco, Robocot |

Se ubican en la **generación 9 ("Propia")** del eje X y se agregó el tipo **Fairy**
al eje Y para las líneas de Mariposon y Pajaroloco.

Ejecuta las celdas en orden: **11.A → 11.B → 11.C** (y opcionalmente 11.D y 11.E).
La celda 11.C necesita que antes hayas corrido la celda que crea `df_pokemon`
(la del dataset de Kaggle o la de PokéAPI).

In [ ]:
# ================================================================
# 11.A  POKÉMON PERSONALIZADOS - REGISTRO DE DATOS
# ================================================================
# Estas son creaciones propias que se integran al mismo esquema de
# datos del dataset original (ID, Pokemon, Tipo, Generación,
# Total Estadísticas e imagen) para poder graficarlas igual que los
# Pokémon oficiales.
# ================================================================

from pathlib import Path

import numpy as np
import pandas as pd

# Carpeta con las imágenes recortadas (debe quedar junto al notebook)
CARPETA_PERSONALIZADOS = Path("pokemon_personalizados")

# Se usa la generación 9 como "generación propia"
GENERACION_PERSONALIZADA = 9

# IDs altos para no chocar con la Pokédex real (1 - 1025)
ID_INICIAL_PERSONALIZADO = 9001

# Cada Pokémon usa las mismas 6 estadísticas base del dataset original.
# Puedes cambiar tipos y stats libremente: el resto del código se adapta.
POKEDEX_PERSONALIZADA = [
    # ---------------- Línea evolutiva 1: Pochoco ----------------
    {
        "Pokemon": "Pochoco",
        "Tipo": "Electric",
        "Tipo Secundario": "Psychic",
        "Familia": "Pochoco",
        "Etapa": "Base",
        "Archivo": "pochoco.png",
        "hp": 50, "attack": 45, "defense": 45,
        "sp_atk": 65, "sp_def": 55, "speed": 60,
    },
    {
        "Pokemon": "Pochotron",
        "Tipo": "Electric",
        "Tipo Secundario": "Psychic",
        "Familia": "Pochoco",
        "Etapa": "Fase 2",
        "Archivo": "pochotron.png",
        "hp": 70, "attack": 65, "defense": 65,
        "sp_atk": 95, "sp_def": 80, "speed": 85,
    },
    {
        "Pokemon": "Pocholord",
        "Tipo": "Electric",
        "Tipo Secundario": "Steel",
        "Familia": "Pochoco",
        "Etapa": "Fase 3",
        "Archivo": "pocholord.png",
        "hp": 80, "attack": 85, "defense": 85,
        "sp_atk": 110, "sp_def": 95, "speed": 90,
    },
    {
        "Pokemon": "Mega Pocholord",
        "Tipo": "Electric",
        "Tipo Secundario": "Steel",
        "Familia": "Pochoco",
        "Etapa": "Mega",
        "Archivo": "mega_pocholord.png",
        "hp": 90, "attack": 110, "defense": 105,
        "sp_atk": 155, "sp_def": 125, "speed": 115,
    },

    # ---------------- Línea evolutiva 2: Mariposon ----------------
    {
        "Pokemon": "Mariposon",
        "Tipo": "Fairy",
        "Tipo Secundario": "Ice",
        "Familia": "Mariposon",
        "Etapa": "Base",
        "Archivo": "mariposon.png",
        "hp": 45, "attack": 40, "defense": 40,
        "sp_atk": 70, "sp_def": 65, "speed": 70,
    },
    {
        "Pokemon": "Maripopon",
        "Tipo": "Fairy",
        "Tipo Secundario": "Ice",
        "Familia": "Mariposon",
        "Etapa": "Fase 2",
        "Archivo": "maripopon.png",
        "hp": 65, "attack": 60, "defense": 60,
        "sp_atk": 100, "sp_def": 90, "speed": 95,
    },
    {
        "Pokemon": "Maripolord",
        "Tipo": "Fairy",
        "Tipo Secundario": "Ice",
        "Familia": "Mariposon",
        "Etapa": "Fase 3",
        "Archivo": "maripolord.png",
        "hp": 80, "attack": 80, "defense": 75,
        "sp_atk": 115, "sp_def": 105, "speed": 95,
    },
    {
        "Pokemon": "Mega Maripolord",
        "Tipo": "Fairy",
        "Tipo Secundario": "Dragon",
        "Familia": "Mariposon",
        "Etapa": "Mega",
        "Archivo": "mega_maripolord.png",
        "hp": 100, "attack": 105, "defense": 95,
        "sp_atk": 150, "sp_def": 135, "speed": 115,
    },

    # ---------------- Especies individuales ----------------
    {
        "Pokemon": "Luneytoons",
        "Tipo": "Water",
        "Tipo Secundario": "Steel",
        "Familia": "Luneytoons",
        "Etapa": "Base",
        "Archivo": "luneytoons.png",
        "hp": 70, "attack": 55, "defense": 85,
        "sp_atk": 60, "sp_def": 75, "speed": 40,
    },
    {
        "Pokemon": "Pajaroloco",
        "Tipo": "Fairy",
        "Tipo Secundario": "Flying",
        "Familia": "Pajaroloco",
        "Etapa": "Base",
        "Archivo": "pajaroloco.png",
        "hp": 55, "attack": 60, "defense": 45,
        "sp_atk": 70, "sp_def": 55, "speed": 95,
    },
    {
        "Pokemon": "Robocot",
        "Tipo": "Electric",
        "Tipo Secundario": "Steel",
        "Familia": "Robocot",
        "Etapa": "Base",
        "Archivo": "robocot.png",
        "hp": 60, "attack": 70, "defense": 55,
        "sp_atk": 85, "sp_def": 60, "speed": 95,
    },
]


# ================================================================
# Construir el DataFrame de personalizados
# ================================================================

COLUMNAS_STATS = ["hp", "attack", "defense", "sp_atk", "sp_def", "speed"]

filas_personalizadas = []

for indice, pokemon in enumerate(POKEDEX_PERSONALIZADA):
    ruta_imagen = CARPETA_PERSONALIZADOS / pokemon["Archivo"]

    total = int(sum(pokemon[stat] for stat in COLUMNAS_STATS))

    fila = {
        "ID": ID_INICIAL_PERSONALIZADO + indice,
        "Pokemon": pokemon["Pokemon"],
        "Tipo": pokemon["Tipo"],
        "Tipo Secundario": pokemon["Tipo Secundario"],
        "Familia": pokemon["Familia"],
        "Etapa": pokemon["Etapa"],
        "Generación": GENERACION_PERSONALIZADA,
        "Total Estadísticas": total,
        "RutaImagen": str(ruta_imagen),
        "Personalizado": True,
    }

    for stat in COLUMNAS_STATS:
        fila[stat] = pokemon[stat]

    filas_personalizadas.append(fila)

df_personalizados = pd.DataFrame(filas_personalizadas)


# ================================================================
# Verificar que las imágenes existan
# ================================================================

imagenes_faltantes = [
    ruta for ruta in df_personalizados["RutaImagen"]
    if not Path(ruta).exists()
]

if imagenes_faltantes:
    print("ATENCIÓN: no se encontraron estas imágenes:")
    for ruta in imagenes_faltantes:
        print("   -", ruta)
    print()
    print("Copia la carpeta 'pokemon_personalizados' junto a este notebook.")
else:
    print("Todas las imágenes personalizadas fueron encontradas.")

print("Pokémon personalizados registrados:", len(df_personalizados))

display(
    df_personalizados[
        ["ID", "Pokemon", "Familia", "Etapa", "Tipo", "Tipo Secundario",
         "Generación", "Total Estadísticas"]
    ]
)

In [ ]:
# ================================================================
# 11.B  INTEGRAR LOS PERSONALIZADOS AL DATASET PRINCIPAL
# ================================================================
# Toma el DataFrame que ya venías usando (df_pokemon) y le agrega
# los Pokémon propios, dejando todo en un solo df_pokemon_total.
# ================================================================

import pandas as pd


# El tipo Fairy no estaba en la lista original: se agrega para que
# la línea de Mariposon y Pajaroloco tengan su propia fila en el eje Y.
TIPOS_ORDEN_TOTAL = [
    "Fire",
    "Water",
    "Grass",
    "Electric",
    "Psychic",
    "Dragon",
    "Normal",
    "Ghost",
    "Fighting",
    "Ice",
    "Fairy",
]

COLORES_TOTAL = {
    "Fire": "#ff5a1f",
    "Water": "#2e86de",
    "Grass": "#44aa44",
    "Electric": "#ffd700",
    "Psychic": "#f06292",
    "Dragon": "#673ab7",
    "Normal": "#b8a382",
    "Ghost": "#5e35b1",
    "Fighting": "#b03a2e",
    "Ice": "#4fc3d9",
    "Fairy": "#ff8fd0",
}

# Etiquetas del eje X (la generación 9 es la propia)
GENERACIONES_EJE = list(range(1, 10))
ETIQUETAS_GENERACION = [str(g) for g in range(1, 9)] + ["Propia"]


def detectar_columna_imagen(df):
    """Encuentra cómo se llama la columna de imagen en el df base."""
    for columna in ["RutaImagen", "Imagen", "Sprite", "image_path"]:
        if columna in df.columns:
            return columna
    return None


def integrar_personalizados(df_base, df_extra):
    """Une el dataset original con los Pokémon personalizados."""

    if df_base is None or len(df_base) == 0:
        print("No hay dataset base disponible. Solo se usarán los personalizados.")
        resultado = df_extra.copy()
        resultado["EsLocal"] = True
        return resultado

    base = df_base.copy()

    columna_imagen = detectar_columna_imagen(base)

    if columna_imagen is None:
        raise ValueError(
            "El DataFrame base no tiene columna de imagen "
            "(se esperaba 'Imagen', 'Sprite' o 'RutaImagen')."
        )

    # Normalizar la columna de imagen a un solo nombre
    base["RutaImagen"] = base[columna_imagen].astype(str)

    # 'EsLocal' indica si la imagen es un archivo del disco o una URL
    base["EsLocal"] = ~base["RutaImagen"].str.startswith(("http://", "https://"))

    base["Personalizado"] = False

    if "Tipo Secundario" not in base.columns:
        base["Tipo Secundario"] = ""

    if "Familia" not in base.columns:
        base["Familia"] = base["Pokemon"]

    if "Etapa" not in base.columns:
        base["Etapa"] = "Oficial"

    extra = df_extra.copy()
    extra["EsLocal"] = True

    columnas = [
        "ID",
        "Pokemon",
        "Tipo",
        "Tipo Secundario",
        "Familia",
        "Etapa",
        "Generación",
        "Total Estadísticas",
        "RutaImagen",
        "EsLocal",
        "Personalizado",
    ]

    base = base[[c for c in columnas if c in base.columns]]
    extra = extra[[c for c in columnas if c in extra.columns]]

    # Evitar duplicados si la celda se ejecuta varias veces
    base = base[~base["Pokemon"].isin(extra["Pokemon"])]

    resultado = pd.concat([base, extra], ignore_index=True)

    resultado["Generación"] = pd.to_numeric(resultado["Generación"], errors="coerce")
    resultado["Total Estadísticas"] = pd.to_numeric(
        resultado["Total Estadísticas"], errors="coerce"
    )

    resultado = resultado.dropna(subset=["Generación", "Total Estadísticas"])

    resultado["Generación"] = resultado["Generación"].astype(int)
    resultado["Total Estadísticas"] = resultado["Total Estadísticas"].astype(int)

    return resultado


# ================================================================
# Ejecutar la integración
# ================================================================

df_base_existente = df_pokemon if "df_pokemon" in globals() else None

df_pokemon_total = integrar_personalizados(df_base_existente, df_personalizados)

print("Pokémon oficiales:", int((~df_pokemon_total["Personalizado"]).sum()))
print("Pokémon personalizados:", int(df_pokemon_total["Personalizado"].sum()))
print("Total en el dataset:", len(df_pokemon_total))
print()

display(
    df_pokemon_total[df_pokemon_total["Personalizado"]][
        ["ID", "Pokemon", "Tipo", "Tipo Secundario", "Etapa",
         "Generación", "Total Estadísticas"]
    ]
)

In [ ]:
# ================================================================
# 11.C  SCATTER 3D CON LOS POKÉMON PERSONALIZADOS INCLUIDOS
# ================================================================

import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import requests
from functools import lru_cache
from PIL import Image

pio.renderers.default = "vscode"


# ---------------- Configuración del gráfico ----------------

TOP_N_POR_TIPO_GENERACION = 3     # Pokémon oficiales por celda tipo/generación
PIXEL_SPRITE = 32                 # Resolución de cada sprite oficial
PIXEL_SPRITE_PERSONALIZADO = 44   # Resolución de los sprites propios

SPRITE_ANCHO_X = 0.70
SPRITE_ALTO_Z = 90

# Los personalizados se dibujan un poco más grandes para que resalten
ESCALA_PERSONALIZADOS = 1.35

CARPETA_SPRITES = "sprites_pokemon"
os.makedirs(CARPETA_SPRITES, exist_ok=True)


# ---------------- Selección de Pokémon a graficar ----------------

df_oficiales = df_pokemon_total[
    (~df_pokemon_total["Personalizado"])
    & (df_pokemon_total["Tipo"].isin(TIPOS_ORDEN_TOTAL))
    & (df_pokemon_total["Generación"].between(1, 8))
].copy()

df_propios = df_pokemon_total[df_pokemon_total["Personalizado"]].copy()

df_oficiales = (
    df_oficiales
    .sort_values(
        ["Tipo", "Generación", "Total Estadísticas"],
        ascending=[True, True, False],
    )
    .groupby(["Tipo", "Generación"], group_keys=False)
    .head(TOP_N_POR_TIPO_GENERACION)
)

# Los personalizados SIEMPRE se incluyen completos
df_plot = pd.concat([df_oficiales, df_propios], ignore_index=True)

mapa_tipo = {tipo: i for i, tipo in enumerate(TIPOS_ORDEN_TOTAL)}
df_plot["tipo_index"] = df_plot["Tipo"].map(mapa_tipo)

df_plot = df_plot.dropna(subset=["tipo_index"]).copy()
df_plot["tipo_index"] = df_plot["tipo_index"].astype(int)

df_plot = df_plot.sort_values(
    ["tipo_index", "Generación", "Total Estadísticas"],
    ascending=[True, True, False],
).reset_index(drop=True)


# ---------------- Separar los que caen en la misma celda ----------------

df_plot["pos_en_celda"] = df_plot.groupby(["Tipo", "Generación"]).cumcount()

offsets = [
    (0.00, 0.00),
    (-0.23, -0.23),
    (0.23, 0.23),
    (-0.23, 0.23),
    (0.23, -0.23),
    (0.00, -0.33),
    (0.00, 0.33),
    (-0.33, 0.00),
    (0.33, 0.00),
    (-0.36, -0.36),
    (0.36, 0.36),
    (-0.36, 0.36),
    (0.36, -0.36),
]

df_plot["offset_x"] = df_plot["pos_en_celda"].apply(lambda i: offsets[i % len(offsets)][0])
df_plot["offset_y"] = df_plot["pos_en_celda"].apply(lambda i: offsets[i % len(offsets)][1])

df_plot["x_plot"] = df_plot["Generación"] + df_plot["offset_x"]
df_plot["y_plot"] = df_plot["tipo_index"] + df_plot["offset_y"]

print("Pokémon en el gráfico:", len(df_plot))
print("   Oficiales:", int((~df_plot["Personalizado"]).sum()))
print("   Personalizados:", int(df_plot["Personalizado"].sum()))


# ---------------- Carga de imágenes ----------------

def obtener_ruta_local(pokemon_id, ruta_o_url, es_local):
    """Devuelve una ruta en disco, descargando el sprite si hace falta."""

    if es_local:
        return ruta_o_url

    destino = os.path.join(CARPETA_SPRITES, f"{int(pokemon_id)}.png")

    if os.path.exists(destino):
        return destino

    try:
        respuesta = requests.get(ruta_o_url, timeout=20)
        respuesta.raise_for_status()

        with open(destino, "wb") as archivo:
            archivo.write(respuesta.content)

        return destino

    except Exception as error:
        print("No se pudo descargar el sprite", pokemon_id, ":", error)
        return None


@lru_cache(maxsize=2000)
def cargar_sprite(pokemon_id, ruta_o_url, es_local, size):
    """Carga la imagen, la deja transparente y la centra en un lienzo cuadrado."""

    ruta = obtener_ruta_local(pokemon_id, ruta_o_url, es_local)

    if ruta is None or not os.path.exists(ruta):
        return None

    try:
        img = Image.open(ruta).convert("RGBA")

        # Quitar fondo casi blanco (por si la imagen no tiene transparencia)
        arr = np.array(img)

        fondo_claro = (
            (arr[:, :, 0] > 246)
            & (arr[:, :, 1] > 246)
            & (arr[:, :, 2] > 246)
        )

        arr[:, :, 3] = np.where(fondo_claro, 0, arr[:, :, 3])
        img = Image.fromarray(arr, mode="RGBA")

        bbox = img.getbbox()
        if bbox:
            img = img.crop(bbox)

        img.thumbnail((size, size), Image.Resampling.LANCZOS)

        lienzo = Image.new("RGBA", (size, size), (0, 0, 0, 0))
        lienzo.paste(
            img,
            ((size - img.width) // 2, (size - img.height) // 2),
            img,
        )

        return np.array(lienzo)

    except Exception as error:
        print("No se pudo procesar la imagen:", ruta, error)
        return None


# ---------------- Convertir cada sprite en una malla 3D ----------------

xs, ys, zs = [], [], []
ii, jj, kk = [], [], []
facecolors = []

indice = 0

for _, fila in df_plot.iterrows():
    es_propio = bool(fila["Personalizado"])

    resolucion = PIXEL_SPRITE_PERSONALIZADO if es_propio else PIXEL_SPRITE

    arr = cargar_sprite(
        int(fila["ID"]),
        str(fila["RutaImagen"]),
        bool(fila["EsLocal"]),
        resolucion,
    )

    if arr is None:
        continue

    escala = ESCALA_PERSONALIZADOS if es_propio else 1.0

    ancho_x = SPRITE_ANCHO_X * escala
    alto_z = SPRITE_ALTO_Z * escala

    centro_x = float(fila["x_plot"])
    centro_y = float(fila["y_plot"])
    centro_z = float(fila["Total Estadísticas"])

    alto, ancho, _ = arr.shape

    pixel_w = ancho_x / ancho
    pixel_h = alto_z / alto

    for py in range(alto):
        for px in range(ancho):
            r, g, b, a = arr[py, px]

            if a < 40:
                continue

            x0 = centro_x - ancho_x / 2 + px * pixel_w
            x1 = x0 + pixel_w

            z1 = centro_z + alto_z / 2 - py * pixel_h
            z0 = z1 - pixel_h

            xs.extend([x0, x1, x1, x0])
            ys.extend([centro_y] * 4)
            zs.extend([z0, z0, z1, z1])

            ii.extend([indice, indice])
            jj.extend([indice + 1, indice + 2])
            kk.extend([indice + 2, indice + 3])

            color = f"rgba({r},{g},{b},{a / 255})"
            facecolors.extend([color, color])

            indice += 4

print("Pixeles 3D generados:", len(xs))

malla_sprites = go.Mesh3d(
    x=xs,
    y=ys,
    z=zs,
    i=ii,
    j=jj,
    k=kk,
    facecolor=facecolors,
    flatshading=True,
    hoverinfo="skip",
    lighting=dict(ambient=1, diffuse=0, specular=0, roughness=1, fresnel=0),
    showscale=False,
    name="Sprites",
)


# ---------------- Figura ----------------

fig = go.Figure()
fig.add_trace(malla_sprites)

PLANTILLA_HOVER = (
    "<b style='font-size:15px'>%{customdata[0]}</b><br><br>"
    "<b>Tipo:</b> %{customdata[1]}<br>"
    "<b>Generación:</b> %{customdata[2]}<br>"
    "<b>Total de estadísticas:</b> %{customdata[3]}<br>"
    "<b>ID:</b> %{customdata[4]}<br>"
    "<b>Etapa:</b> %{customdata[5]}"
    "<extra></extra>"
)


def construir_customdata(datos):
    return np.stack(
        [
            datos["Pokemon"],
            datos["Tipo"] + np.where(
                datos["Tipo Secundario"].fillna("").astype(str) != "",
                " / " + datos["Tipo Secundario"].fillna("").astype(str),
                "",
            ),
            datos["Generación"],
            datos["Total Estadísticas"],
            datos["ID"],
            datos["Etapa"],
        ],
        axis=-1,
    )


# Puntos de hover de los Pokémon oficiales
for tipo in TIPOS_ORDEN_TOTAL:
    datos_tipo = df_plot[
        (df_plot["Tipo"] == tipo) & (~df_plot["Personalizado"])
    ].copy()

    if datos_tipo.empty:
        continue

    fig.add_trace(
        go.Scatter3d(
            x=datos_tipo["x_plot"],
            y=datos_tipo["y_plot"],
            z=datos_tipo["Total Estadísticas"],
            mode="markers",
            name=tipo,
            customdata=construir_customdata(datos_tipo),
            marker=dict(size=9, color=COLORES_TOTAL[tipo], opacity=0.13),
            hovertemplate=PLANTILLA_HOVER,
        )
    )

# Puntos de hover de los personalizados (con anillo visible)
if not df_propios.empty:
    datos_propios = df_plot[df_plot["Personalizado"]].copy()

    fig.add_trace(
        go.Scatter3d(
            x=datos_propios["x_plot"],
            y=datos_propios["y_plot"],
            z=datos_propios["Total Estadísticas"] - SPRITE_ALTO_Z * ESCALA_PERSONALIZADOS / 2,
            mode="markers",
            name="Personalizados",
            customdata=construir_customdata(datos_propios),
            marker=dict(
                size=7,
                color="#7c3aed",
                opacity=0.85,
                symbol="diamond",
                line=dict(color="white", width=1),
            ),
            hovertemplate=PLANTILLA_HOVER,
        )
    )

    # Línea vertical que ancla cada creación propia al piso del gráfico
    z_piso = df_plot["Total Estadísticas"].min() - 60

    for _, fila in datos_propios.iterrows():
        fig.add_trace(
            go.Scatter3d(
                x=[fila["x_plot"], fila["x_plot"]],
                y=[fila["y_plot"], fila["y_plot"]],
                z=[z_piso, fila["Total Estadísticas"] - 30],
                mode="lines",
                line=dict(color="rgba(124,58,237,0.35)", width=2),
                hoverinfo="skip",
                showlegend=False,
            )
        )


z_min = int(np.floor(df_plot["Total Estadísticas"].min() / 50) * 50) - 80
z_max = int(np.ceil(df_plot["Total Estadísticas"].max() / 50) * 50) + 80

fig.update_layout(
    title=dict(
        text=(
            "Scatter 3D de Pokémon por Tipo y Generación "
            "<span style='color:#7c3aed'>(incluye creaciones propias)</span>"
        ),
        x=0.02,
        y=0.95,
        font=dict(size=22, color="#22345b"),
    ),
    width=1250,
    height=850,
    margin=dict(l=0, r=0, t=70, b=0),
    legend=dict(
        title="Tipo",
        x=0.86,
        y=0.85,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#d9e1ea",
        borderwidth=1,
    ),
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#cbd5e1",
        font=dict(size=13, color="#22345b"),
    ),
    scene=dict(
        xaxis=dict(
            title="Generación",
            tickmode="array",
            tickvals=GENERACIONES_EJE,
            ticktext=ETIQUETAS_GENERACION,
            range=[0.4, 9.7],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False,
        ),
        yaxis=dict(
            title="Tipo",
            tickmode="array",
            tickvals=list(range(len(TIPOS_ORDEN_TOTAL))),
            ticktext=TIPOS_ORDEN_TOTAL,
            range=[len(TIPOS_ORDEN_TOTAL) - 0.7, -0.7],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False,
        ),
        zaxis=dict(
            title="Total de estadísticas",
            range=[z_min, z_max],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
            zerolinecolor="white",
            showspikes=False,
        ),
        camera=dict(eye=dict(x=1.65, y=-2.15, z=0.95)),
        aspectmode="manual",
        aspectratio=dict(x=1.5, y=1.15, z=1.0),
    ),
)

fig.show()

In [ ]:
# ================================================================
# 11.D  VERSIÓN MATPLOTLIB (imagen estática) CON LOS PERSONALIZADOS
# ================================================================
# Usa la misma técnica de sprites en "figure pixels" que ya te
# funcionaba en VS Code.
# ================================================================

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from mpl_toolkits.mplot3d import proj3d

SALIDA_IMAGEN = "pokemon_3d_con_personalizados.png"

plt.rcParams["figure.dpi"] = 120

fig_mpl = plt.figure(figsize=(17, 10))
ax = fig_mpl.add_subplot(111, projection="3d")
ax.set_position([0.04, 0.08, 0.72, 0.82])

# Puntos base (por si algún sprite falla)
ax.scatter(
    df_plot["x_plot"],
    df_plot["y_plot"],
    df_plot["Total Estadísticas"],
    s=np.where(df_plot["Personalizado"], 110, 70),
    alpha=0.20,
    c=[
        "#7c3aed" if propio else COLORES_TOTAL.get(tipo, "gray")
        for propio, tipo in zip(df_plot["Personalizado"], df_plot["Tipo"])
    ],
    depthshade=True,
)

ax.set_title(
    "Scatter 3D de Pokémon por Tipo y Generación (incluye creaciones propias)",
    fontsize=15,
    fontweight="bold",
    pad=25,
)

ax.set_xlabel("Generación", fontsize=12, labelpad=14)
ax.set_ylabel("Tipo", fontsize=12, labelpad=14)
ax.set_zlabel("Total de estadísticas", fontsize=12, labelpad=14)

ax.set_xlim(0.5, 9.6)
ax.set_xticks(GENERACIONES_EJE)
ax.set_xticklabels(ETIQUETAS_GENERACION)

ax.set_ylim(len(TIPOS_ORDEN_TOTAL) - 0.5, -0.5)
ax.set_yticks(range(len(TIPOS_ORDEN_TOTAL)))
ax.set_yticklabels(TIPOS_ORDEN_TOTAL)

z_min = int(np.floor(df_plot["Total Estadísticas"].min() / 50) * 50)
z_max = int(np.ceil(df_plot["Total Estadísticas"].max() / 50) * 50)
ax.set_zlim(z_min - 60, z_max + 60)

ax.view_init(elev=22, azim=-62)
ax.set_box_aspect((9, 6, 4.6))
ax.grid(True)

try:
    ax.xaxis.pane.set_alpha(0.12)
    ax.yaxis.pane.set_alpha(0.12)
    ax.zaxis.pane.set_alpha(0.12)
except Exception:
    pass


# ---------------- Leyenda ----------------

handles_leyenda = [
    Line2D([0], [0], marker="o", color="w", label=tipo,
           markerfacecolor=COLORES_TOTAL[tipo], markersize=10)
    for tipo in TIPOS_ORDEN_TOTAL
    if tipo in df_plot["Tipo"].values
]

handles_leyenda.append(
    Line2D([0], [0], marker="D", color="w", label="Creación propia",
           markerfacecolor="#7c3aed", markersize=11)
)

ax.legend(
    handles=handles_leyenda,
    title="Tipo",
    loc="upper left",
    bbox_to_anchor=(1.05, 0.95),
)


# ---------------- Sprites sobre el gráfico ----------------

fig_mpl.canvas.draw()

for _, fila in df_plot.iterrows():
    es_propio = bool(fila["Personalizado"])

    arr = cargar_sprite(
        int(fila["ID"]),
        str(fila["RutaImagen"]),
        bool(fila["EsLocal"]),
        110 if es_propio else 72,
    )

    if arr is None:
        continue

    x2, y2, z2 = proj3d.proj_transform(
        fila["x_plot"],
        fila["y_plot"],
        fila["Total Estadísticas"],
        ax.get_proj(),
    )

    xpix, ypix = ax.transData.transform((x2, y2))

    caja_imagen = OffsetImage(
        arr,
        zoom=0.62 if es_propio else 0.50,
        resample=True,
    )

    anotacion = AnnotationBbox(
        caja_imagen,
        (xpix, ypix),
        xycoords="figure pixels",
        frameon=False,
        pad=0.0,
        box_alignment=(0.5, 0.5),
        annotation_clip=False,
    )

    anotacion.set_zorder(int(10000 - z2 * 1000))
    fig_mpl.add_artist(anotacion)

    # Nombre debajo de cada creación propia
    if es_propio:
        fig_mpl.text(
            xpix,
            ypix - 44,
            fila["Pokemon"],
            transform=None,
            fontsize=8,
            color="#5b21b6",
            fontweight="bold",
            ha="center",
            zorder=20000,
        )


fig_mpl.savefig(SALIDA_IMAGEN, dpi=160)
print("Imagen guardada como:", SALIDA_IMAGEN)

plt.show()

In [ ]:
# ================================================================
# 11.E  GRÁFICO 3D DE LAS LÍNEAS EVOLUTIVAS PROPIAS
# ================================================================
# Eje X = etapa evolutiva, Eje Y = familia, Eje Z = total de stats.
# ================================================================

import numpy as np
import plotly.graph_objects as go

ORDEN_ETAPAS = ["Base", "Fase 2", "Fase 3", "Mega"]

df_evo = df_personalizados.copy()
df_evo["etapa_index"] = df_evo["Etapa"].map({e: i for i, e in enumerate(ORDEN_ETAPAS)})
df_evo = df_evo.dropna(subset=["etapa_index"])
df_evo["etapa_index"] = df_evo["etapa_index"].astype(int)

familias = df_evo["Familia"].drop_duplicates().tolist()
mapa_familia = {familia: i for i, familia in enumerate(familias)}
df_evo["familia_index"] = df_evo["Familia"].map(mapa_familia)

df_evo = df_evo.sort_values(["familia_index", "etapa_index"]).reset_index(drop=True)


# ---------------- Sprites como malla 3D ----------------

ANCHO_X_EVO = 0.55
ALTO_Z_EVO = 95
PIXELES_EVO = 48

xs, ys, zs = [], [], []
ii, jj, kk = [], [], []
facecolors = []
indice = 0

for _, fila in df_evo.iterrows():
    arr = cargar_sprite(int(fila["ID"]), str(fila["RutaImagen"]), True, PIXELES_EVO)

    if arr is None:
        continue

    centro_x = float(fila["etapa_index"])
    centro_y = float(fila["familia_index"])
    centro_z = float(fila["Total Estadísticas"])

    alto, ancho, _ = arr.shape

    pixel_w = ANCHO_X_EVO / ancho
    pixel_h = ALTO_Z_EVO / alto

    for py in range(alto):
        for px in range(ancho):
            r, g, b, a = arr[py, px]

            if a < 40:
                continue

            x0 = centro_x - ANCHO_X_EVO / 2 + px * pixel_w
            x1 = x0 + pixel_w

            z1 = centro_z + ALTO_Z_EVO / 2 - py * pixel_h
            z0 = z1 - pixel_h

            xs.extend([x0, x1, x1, x0])
            ys.extend([centro_y] * 4)
            zs.extend([z0, z0, z1, z1])

            ii.extend([indice, indice])
            jj.extend([indice + 1, indice + 2])
            kk.extend([indice + 2, indice + 3])

            color = f"rgba({r},{g},{b},{a / 255})"
            facecolors.extend([color, color])

            indice += 4


fig_evo = go.Figure()

fig_evo.add_trace(
    go.Mesh3d(
        x=xs, y=ys, z=zs,
        i=ii, j=jj, k=kk,
        facecolor=facecolors,
        flatshading=True,
        hoverinfo="skip",
        lighting=dict(ambient=1, diffuse=0, specular=0, roughness=1, fresnel=0),
        showscale=False,
        name="Sprites propios",
    )
)

# Línea de progresión por familia
paleta = ["#7c3aed", "#ec4899", "#0ea5e9", "#f59e0b", "#10b981"]

for familia, indice_familia in mapa_familia.items():
    datos = df_evo[df_evo["Familia"] == familia]

    fig_evo.add_trace(
        go.Scatter3d(
            x=datos["etapa_index"],
            y=datos["familia_index"],
            z=datos["Total Estadísticas"],
            mode="lines+markers",
            name=familia,
            line=dict(color=paleta[indice_familia % len(paleta)], width=5),
            marker=dict(size=5, color=paleta[indice_familia % len(paleta)]),
            customdata=np.stack(
                [
                    datos["Pokemon"],
                    datos["Etapa"],
                    datos["Tipo"] + " / " + datos["Tipo Secundario"],
                    datos["Total Estadísticas"],
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b style='font-size:15px'>%{customdata[0]}</b><br><br>"
                "<b>Etapa:</b> %{customdata[1]}<br>"
                "<b>Tipo:</b> %{customdata[2]}<br>"
                "<b>Total de estadísticas:</b> %{customdata[3]}"
                "<extra></extra>"
            ),
        )
    )

fig_evo.update_layout(
    title=dict(
        text="Líneas evolutivas propias: progresión de estadísticas",
        x=0.02,
        y=0.95,
        font=dict(size=22, color="#22345b"),
    ),
    width=1150,
    height=760,
    margin=dict(l=0, r=0, t=70, b=0),
    hoverlabel=dict(bgcolor="white", bordercolor="#cbd5e1", font=dict(size=13)),
    scene=dict(
        xaxis=dict(
            title="Etapa evolutiva",
            tickmode="array",
            tickvals=list(range(len(ORDEN_ETAPAS))),
            ticktext=ORDEN_ETAPAS,
            range=[-0.6, len(ORDEN_ETAPAS) - 0.4],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
        ),
        yaxis=dict(
            title="Familia",
            tickmode="array",
            tickvals=list(range(len(familias))),
            ticktext=familias,
            range=[len(familias) - 0.4, -0.6],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
        ),
        zaxis=dict(
            title="Total de estadísticas",
            range=[
                df_evo["Total Estadísticas"].min() - 90,
                df_evo["Total Estadísticas"].max() + 90,
            ],
            backgroundcolor="rgb(245,248,252)",
            gridcolor="white",
        ),
        camera=dict(eye=dict(x=1.55, y=-1.95, z=0.85)),
        aspectmode="manual",
        aspectratio=dict(x=1.3, y=1.0, z=0.95),
    ),
)

fig_evo.show()